# UPAR Challenge 2027, Track 1: pipeline notebook

This notebook has the code we ran on Colab (free T4), in the order you would run it: setup and data, analysis, image
cache, training, experiments, and the Codabench package. The long training runs were done in separate Colab sessions,
one run each, and their commands are listed in Part 4 with the logs summarised in `results/`. A few outputs are missing
where we re-ran a cell in a fresh session, so treat the code as the record and `results/` as the numbers.

Use a T4 runtime, with Google Drive mounted at `/content/drive` and a folder `MyDrive/upar`.

## Part 1. Setup and data

Mount Drive, get the organisers' repository, download Market1501, PA-100K and PETA, and check that every image in the annotation files exists.

### GPU and versions

In [ ]:
!nvidia-smi
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| cuda available:', torch.cuda.is_available())
print('Grader runs torch 2.4.1 / torchvision 0.19.1. Avoid newer APIs.')
if not torch.cuda.is_available():
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU (this restarts the VM). Data setup below still works without a GPU.')

### Drive and paths

In [ ]:
import os, shutil, zipfile, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/upar')
for sub in ('', 'code', 'archives', 'manual', 'verify', 'checkpoints'):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
DATA = Path('/content/data'); DATA.mkdir(exist_ok=True)

u = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive reports: total {u.total/1e9:.1f} GB, free {u.free/1e9:.1f} GB (you said ~7 GB free)')
print('Local disk free:', f'{shutil.disk_usage("/content").free/1e9:.1f} GB')

### Repository

The zip of the organisers' repository (it has `download_datasets.py` and the annotation CSVs) sits in `MyDrive/upar/`.

In [ ]:
REPO_ZIP = DRIVE / 'UPAR-Challenge-2027-main.zip'
if not REPO_ZIP.exists():
    cands = [p for p in DRIVE.glob('*.zip') if 'upar' in p.name.lower()]
    assert cands, f'Upload the repo zip to {DRIVE} first'
    REPO_ZIP = cands[0]

SRC = Path('/content/repo_src')
found = list(SRC.rglob('download_datasets.py')) if SRC.exists() else []
if not found:
    shutil.rmtree(SRC, ignore_errors=True)
    with zipfile.ZipFile(REPO_ZIP) as zf:
        zf.extractall(SRC)
    found = list(SRC.rglob('download_datasets.py'))
assert found, 'download_datasets.py not found inside the zip'
REPO = found[0].parent
need = ['peta_file_mapping.txt', 'data/annotations/task1/train/gt.csv', 'data/annotations/task1/val/gt.csv']
missing = [n for n in need if not (REPO / n).exists()]
print('REPO =', REPO, '| missing files:', missing or 'none')
assert not missing

### Install

In [ ]:
!pip install -q -U gdown tqdm
import gdown; print('gdown', gdown.__version__)

### Download the datasets

The organisers' script stops at the first failure, so this cell calls its functions one dataset at a time and skips what is already there.

In [ ]:
os.chdir(REPO)                      # download_datasets.py reads peta_file_mapping.txt from the cwd
sys.path.insert(0, str(REPO))
import download_datasets as dl

shutil.copytree(REPO / 'data' / 'annotations', DATA / 'annotations', dirs_exist_ok=True)

MARKET_URL = "https://drive.google.com/file/d/0B8-rUzbwVRk0c054eEozWG9COHM/view?resourcekey=0-8nyl7K9_x37HlQm34MmrYQ"
PA100K_URL = "https://drive.google.com/drive/folders/1d_D0Yh7C262gr0ef9EqkvG_M3fqgAWa2?usp=sharing"
PETA_URL   = "https://www.dropbox.com/s/52ylx522hwbdxz6/PETA.zip?dl=1"

ARCHIVES = {'market': DATA / 'market_1501.zip',
            'pa100k': DATA / 'PA100k' / 'data.zip',
            'peta':   DATA / 'PETA' / 'peta.zip'}
def drive_copy(key): return DRIVE / 'archives' / ARCHIVES[key].name
def have_zip(p): return p.exists() and zipfile.is_zipfile(p)
def count(p):
    try: return sum(1 for _ in os.scandir(p))
    except FileNotFoundError: return 0

DONE = {
    'market': lambda: count(DATA/'Market1501'/'bounding_box_train') > 0 and count(DATA/'Market1501'/'query') > 0,
    'pa100k': lambda: count(DATA/'PA100k'/'release_data'/'release_data') >= 99_000,
    'peta':   lambda: count(DATA/'PETA'/'images') >= 18_900,
}

def fetch_archive(key, download):
    local = ARCHIVES[key]; local.parent.mkdir(parents=True, exist_ok=True)
    if local.exists() and not zipfile.is_zipfile(local):
        print(f'  {local.name}: incomplete file, deleting'); local.unlink()
    if not local.exists() and have_zip(drive_copy(key)):
        print(f'  {key}: restoring archive from Drive'); shutil.copyfile(drive_copy(key), local)
    if not local.exists():
        download()
    if not have_zip(local):
        raise RuntimeError(f'{key}: no valid zip at {local} (download failed)')

def gdl(url, out):
    # gdown >= 6 dropped the fuzzy= argument (the repo's own script still passes it)
    try:
        return gdown.download(url, output=out, quiet=False, use_cookies=False, fuzzy=True)
    except TypeError:
        return gdown.download(url, output=out, quiet=False, use_cookies=False)

def step_market():
    fetch_archive('market', lambda: gdl(MARKET_URL, str(ARCHIVES['market'])))
    raw = DATA / 'Market-1501-v15.09.15'
    if not raw.exists(): dl.extract_zip(ARCHIVES['market'], DATA)
    shutil.rmtree(DATA / 'Market1501', ignore_errors=True)
    raw.rename(DATA / 'Market1501')

def step_pa100k():
    fetch_archive('pa100k', lambda: gdown.download_folder(PA100K_URL, output=str(DATA/'PA100k'),
                                                          quiet=False, use_cookies=False))
    dl.extract_zip(ARCHIVES['pa100k'], DATA / 'PA100k')

def step_peta():
    fetch_archive('peta', lambda: dl.download_url(PETA_URL, ARCHIVES['peta']))
    dl.prepare_peta(DATA)           # skips the download (zip exists), extracts, renames via the mapping

STEPS = {'market': step_market, 'pa100k': step_pa100k, 'peta': step_peta}

def run_step(key):
    if DONE[key]():
        print(f'[skip] {key}: already in {DATA}'); return True
    print(f'[run ] {key}')
    try:
        STEPS[key]()
    except Exception as e:
        print(f'[FAIL] {key}: {type(e).__name__}: {str(e)[:500]}')
        print('  Google Drive may be over its download quota: wait, or copy the file to your own Drive and unzip it into /content/data.')
    ok = DONE[key]()
    print(('[ ok ] ' if ok else '[FAIL] ') + key)
    return ok

results = {k: run_step(k) for k in STEPS}      # one failure does not stop the others
print('\nSummary:', results)

### Disk space and sizes

In [ ]:
import subprocess
u = shutil.disk_usage('/content')
print(f'/content free: {u.free/1e9:.1f} GB of {u.total/1e9:.1f} GB\n')
for name in ['Market1501', 'PA100k', 'PETA']:
    p = DATA / name
    if p.exists():
        size = subprocess.run(['du', '-sh', str(p)], capture_output=True, text=True).stdout.split()[0]
        print(f'{name:11s} {size:>7s}')
    else:
        print(f'{name:11s} MISSING')
print()
for k, z in ARCHIVES.items():
    print(f'archive {z.name:16s}', f'{z.stat().st_size/1e9:.2f} GB' if z.exists() else 'not present')

### Verify every image

Checks that each path in `gt.csv` exists and opens, prints the counts per source and the image sizes.

In [ ]:
%%writefile /content/drive/MyDrive/upar/code/verify_data.py
"""Verify UPAR task-1 annotations against the images on disk."""
import argparse, csv, json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

EXPECTED = {
    "train": {"total": 97669, "PA100k": 79001, "Market1501": 10000, "PETA": 8668},
    "val":   {"total": 33407, "Market1501": 16458, "PA100k": 9986, "PETA": 6963},
}


def read_split(csv_path):
    with open(csv_path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = [r for r in reader if r]
    problems = []
    if header[0] != "# image" or len(header) != 41:
        problems.append(f"unexpected header: first={header[0]!r}, n_cols={len(header)}")
    ok_vals = {"0", "1", "0.0", "1.0"}
    bad = sum(1 for r in rows if len(r) != 41 or any(v not in ok_vals for v in r[1:]))
    if bad:
        problems.append(f"{bad} rows with wrong length or values other than 0/1")
    return [r[0] for r in rows], problems


def check_image(job):
    root, rel, full = job
    p = root / rel
    if not p.is_file():
        return rel, "missing", None
    try:
        with Image.open(p) as im:
            size = im.size
            if full:
                im.load()
        return rel, "ok", size
    except Exception as e:
        return rel, f"unreadable: {type(e).__name__}: {e}", None


def src_of(rel):
    return rel.split("/")[0]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default="./data")
    ap.add_argument("--out-dir", default="./verify_out")
    ap.add_argument("--full", action="store_true", help="decode every image (slow)")
    ap.add_argument("--workers", type=int, default=32)
    args = ap.parse_args()
    root, out = Path(args.data_dir), Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)

    splits, issues = {}, []
    for split in ("train", "val"):
        paths, problems = read_split(root / "annotations" / "task1" / split / "gt.csv")
        splits[split] = paths
        issues += [f"{split}: {p}" for p in problems]
        counts = {}
        for p in paths:
            counts[src_of(p)] = counts.get(src_of(p), 0) + 1
        print(f"\n[{split}] {len(paths)} rows (expected {EXPECTED[split]['total']})")
        for s, n in sorted(counts.items()):
            exp = EXPECTED[split].get(s)
            flag = "" if exp == n else f"   <-- expected {exp}"
            print(f"   {s:11s} {n:6d}{flag}")
        if len(paths) != EXPECTED[split]["total"]:
            issues.append(f"{split}: row count {len(paths)} != {EXPECTED[split]['total']}")
        dup = len(paths) - len(set(paths))
        if dup:
            issues.append(f"{split}: {dup} duplicate paths")

    overlap = set(splits["train"]) & set(splits["val"])
    print(f"\ntrain/val path overlap: {len(overlap)} (expected 0)")
    if overlap:
        issues.append(f"{len(overlap)} paths in both train and val")

    unique = sorted(set(splits["train"]) | set(splits["val"]))
    jobs = [(root, rel, args.full) for rel in unique]
    with ThreadPoolExecutor(args.workers) as ex:
        results = list(tqdm(ex.map(check_image, jobs), total=len(jobs), desc="checking images"))

    missing = [r for r, s, _ in results if s == "missing"]
    bad = [(r, s) for r, s, _ in results if s not in ("ok", "missing")]
    (out / "missing_files.txt").write_text("\n".join(missing))
    (out / "unreadable_files.txt").write_text("\n".join(f"{r}\t{s}" for r, s in bad))

    sizes = {}
    for rel, status, size in results:
        if status == "ok":
            sizes.setdefault(src_of(rel), []).append(size)

    stats = {}
    print("\nImage size statistics (width x height, pixels)")
    print(f"{'source':11s} {'n':>6s}  {'W min/p5/med/p95/max':>26s}  {'H min/p5/med/p95/max':>26s}  med H/W")
    for s, lst in sorted(sizes.items()):
        a = np.array(lst)
        w, h = a[:, 0], a[:, 1]
        q = lambda x: [int(x.min()), int(np.percentile(x, 5)), int(np.median(x)), int(np.percentile(x, 95)), int(x.max())]
        stats[s] = {"n": len(lst), "w": q(w), "h": q(h), "median_h_over_w": float(np.median(h / w))}
        print(f"{s:11s} {len(lst):6d}  {'/'.join(map(str, stats[s]['w'])):>26s}  "
              f"{'/'.join(map(str, stats[s]['h'])):>26s}  {stats[s]['median_h_over_w']:.2f}")
    (out / "image_stats.json").write_text(json.dumps(stats, indent=2))

    print(f"\nunique images: {len(unique)} | missing: {len(missing)} | unreadable: {len(bad)}")
    for i in issues:
        print("ISSUE:", i)
    print(f"reports saved in {out}")
    if missing or bad or issues:
        print("RESULT: PROBLEMS FOUND")
        raise SystemExit(1)
    print("RESULT: ALL OK")


if __name__ == "__main__":
    main()

In [ ]:
!python /content/drive/MyDrive/upar/code/verify_data.py --data-dir /content/data --out-dir /content/drive/MyDrive/upar/verify

### Keep the archives on Drive

Saved only if they fit, so a new session can restore the data without downloading it again.

In [ ]:
RESERVE_GB = 4.0          # keep this free on Drive for checkpoints + image cache
DRIVE_FREE_CAP_GB = 7.0   # your stated free space; guards against a wrong Drive report
free_gb = min(shutil.disk_usage('/content/drive/MyDrive').free / 1e9, DRIVE_FREE_CAP_GB)
print(f'Drive free (used for decision): {free_gb:.2f} GB, reserve {RESERVE_GB} GB\n')

re_download = []
for key in ['market', 'pa100k', 'peta']:
    src, dst = ARCHIVES[key], drive_copy(key)
    if not src.exists():
        print(f'{key}: no local archive, skipping'); continue
    size = src.stat().st_size / 1e9
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print(f'{key}: already on Drive ({size:.2f} GB)'); continue
    if size + RESERVE_GB <= free_gb:
        tmp = dst.with_suffix('.part')
        shutil.copyfile(src, tmp); tmp.rename(dst)
        free_gb -= size
        print(f'{key}: saved to Drive ({size:.2f} GB), free now ~{free_gb:.2f} GB')
    else:
        re_download.append(key)
        print(f'{key}: {size:.2f} GB does NOT fit with the reserve -> not saved')

if re_download:
    print('\nEach new session this notebook will re-download:', re_download)
    print('Google-Drive-sourced ones can hit the quota error again; keep the manual/ copies if you made them.')
else:
    print('\nAll archives are on Drive: a new session restores them without any download.')

## Part 2. Data analysis and the official metric

These findings shaped the loss: masking, independent sigmoids and source-balanced sampling.

### Load the annotations

In [ ]:
import zipfile, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/upar')
FIG = DRIVE / 'figures'; FIG.mkdir(parents=True, exist_ok=True)

ANN = Path('/content/data/annotations/task1')
if not (ANN / 'train' / 'gt.csv').exists():           # fresh session: annotations are inside the repo zip on Drive
    repo_zip = next(p for p in DRIVE.glob('*.zip') if 'upar' in p.name.lower())
    with zipfile.ZipFile(repo_zip) as zf:
        zf.extractall('/content/repo_src')
    ANN = next(Path('/content/repo_src').glob('**/data/annotations/task1'))

tr, va = pd.read_csv(ANN / 'train' / 'gt.csv'), pd.read_csv(ANN / 'val' / 'gt.csv')
NAMES = list(tr.columns[1:]); assert len(NAMES) == 40 and NAMES == list(va.columns[1:])
SOURCES = ['Market1501', 'PA100k', 'PETA']
ytr, yva = tr[NAMES].to_numpy(np.uint8), va[NAMES].to_numpy(np.uint8)
src_tr = tr.iloc[:, 0].str.split('/').str[0].to_numpy()
src_va = va.iloc[:, 0].str.split('/').str[0].to_numpy()
assert set(src_tr) == set(src_va) == set(SOURCES)
print('loaded', len(tr), 'train and', len(va), 'val rows')

### Counts per source

In [ ]:
print('label values present:', sorted(np.unique(np.concatenate([ytr.ravel(), yva.ravel()])).tolist()))
rows = []
for split, y, s in (('train', ytr, src_tr), ('val', yva, src_va)):
    for src in SOURCES:
        m = s == src
        rows.append({'split': split, 'source': src, 'images': int(m.sum()),
                     'positives per image (mean)': round(float(y[m].sum(1).mean()), 2),
                     'images with 0 positives': int((y[m].sum(1) == 0).sum())})
display(pd.DataFrame(rows))

### Positive rate, train against validation

In [ ]:
rate_tr, rate_va = ytr.mean(0), yva.mean(0)
y = np.arange(40)
fig, ax = plt.subplots(figsize=(9, 13))
ax.barh(y - 0.2, rate_tr * 100, height=0.4, label='train')
ax.barh(y + 0.2, rate_va * 100, height=0.4, label='val')
ax.set_yticks(y); ax.set_yticklabels(NAMES, fontsize=8); ax.invert_yaxis()
ax.axvline(2, color='k', ls='--', lw=0.8); ax.text(2.3, 39.5, '2%', fontsize=8)
ax.set_xlabel('positive rate (% of images)')
ax.set_title('Per-attribute positive rate, train vs val (sources pooled)')
ax.legend(); plt.tight_layout()
fig.savefig(FIG / 'fig1_positive_rate_train_vs_val.png', dpi=200); plt.show()

gap = pd.DataFrame({'attribute': NAMES, 'train %': rate_tr * 100, 'val %': rate_va * 100})
gap['abs diff'] = (gap['train %'] - gap['val %']).abs()
print('largest train-vs-val differences:')
display(gap.sort_values('abs diff', ascending=False).head(10).round(2))

### Positive rate per source

In [ ]:
cols = [(sp, s) for sp in ('train', 'val') for s in SOURCES]
M = np.stack([(ytr if sp == 'train' else yva)[(src_tr if sp == 'train' else src_va) == s].mean(0)
              for sp, s in cols], axis=1)                       # 40 x 6
fig, ax = plt.subplots(figsize=(8, 14))
ax.imshow(M, aspect='auto', cmap='Blues', vmin=0, vmax=1)
for i in range(40):
    for j in range(6):
        v = M[i, j]
        t = '-' if v == 0 else ('<1' if v < 0.01 else f'{v * 100:.0f}')
        ax.text(j, i, t, ha='center', va='center', fontsize=7, color='white' if v > 0.55 else 'black')
ax.set_xticks(range(6)); ax.set_xticklabels([f'{sp}\n{s}' for sp, s in cols], fontsize=8)
ax.set_yticks(range(40)); ax.set_yticklabels(NAMES, fontsize=8); ax.xaxis.tick_top()
ax.axvline(2.5, color='k', lw=1)
ax.set_title('Positive rate (%) per source   (- = never positive)', pad=34)
plt.tight_layout(); fig.savefig(FIG / 'fig2_positive_rate_by_source.png', dpi=200); plt.show()

### Never-annotated and rare attributes

In [ ]:
zero_rows = []
for sp, y_, s in (('train', ytr, src_tr), ('val', yva, src_va)):
    for src in SOURCES:
        m = s == src
        for k in np.where(y_[m].sum(0) == 0)[0]:
            zero_rows.append({'split': sp, 'source': src, 'attribute': NAMES[k]})
zdf = pd.DataFrame(zero_rows, columns=['split', 'source', 'attribute'])
if len(zdf):
    show = zdf.groupby(['source', 'attribute'])['split'].apply(lambda x: '+'.join(sorted(x))).reset_index()
    display(show.rename(columns={'split': 'all-zero in'}))
else:
    print('no all-zero attributes in any source')
zdf.to_csv(FIG / 'all_zero_attributes_by_source.csv', index=False)

rare = [NAMES[k] for k in np.where(rate_tr < 0.02)[0]]
print(f'\n{len(rare)} attributes with < 2% positives in train (your note says 12):\n', rare)
for src in SOURCES:
    m = src_tr == src; r = ytr[m].mean(0)
    print(f'{src}: attributes with 0 < rate < 0.5% ->', [(NAMES[k], round(r[k] * 100, 2)) for k in np.where((r > 0) & (r < 0.005))[0]])

### Images with no label in an exclusive group

In [ ]:
GROUPS = {'Age': 'Age-', 'Hair-Length': 'Hair-Length-', 'UpperBody-Color': 'UpperBody-Color-',
          'LowerBody-Color': 'LowerBody-Color-', 'LowerBody-Type': 'LowerBody-Type-'}
GIDX = {g: [i for i, n in enumerate(NAMES) if n.startswith(p)] for g, p in GROUPS.items()}
print('group sizes (expect 3, 3, 12, 12, 2):', {g: len(i) for g, i in GIDX.items()})

rows = []
for sp, y_, s in (('train', ytr, src_tr), ('val', yva, src_va)):
    for src in SOURCES:
        yy = y_[s == src]
        r = {'split': sp, 'source': src, 'images': len(yy)}
        for g, idx in GIDX.items():
            c = yy[:, idx].sum(1)
            r[g + ' | none %'] = round(100 * float((c == 0).mean()), 1)
            r[g + ' | multi %'] = round(100 * float((c > 1).mean()), 1)
        rows.append(r)
gdf = pd.DataFrame(rows)
gdf.to_csv(FIG / 'exclusive_group_stats.csv', index=False)
print('\n% of images with NO positive in the group:')
display(gdf[['split', 'source', 'images'] + [c for c in gdf.columns if c.endswith('none %')]])
print('% of images with MORE THAN ONE positive in the group:')
display(gdf[['split', 'source', 'images'] + [c for c in gdf.columns if c.endswith('multi %')]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, sp in zip(axes, ('train', 'val')):
    sub = gdf[gdf.split == sp]
    for k, src in enumerate(SOURCES):
        vals = [sub[sub.source == src][g + ' | none %'].iloc[0] for g in GROUPS]
        ax.bar(np.arange(len(GROUPS)) + (k - 1) * 0.27, vals, width=0.27, label=src)
    ax.set_xticks(range(len(GROUPS))); ax.set_xticklabels(list(GROUPS), rotation=20, fontsize=8)
    ax.set_title(f'{sp}: images with no label in the group'); ax.set_ylabel('% of images')
axes[0].legend(); plt.tight_layout(); fig.savefig(FIG / 'fig3_no_label_by_group.png', dpi=200); plt.show()

### The metric

Our re-implementation of the official scoring (per dataset, then averaged). It reproduces the 0.547 / 0.500 / 0.605 we measured for the training-prior baseline.

In [ ]:
%%writefile /content/drive/MyDrive/upar/code/upar_eval.py
"""Re-implementation of the UPAR Track 1 metrics as described on the Codabench Evaluation page.
Assumption (not stated on the page): when a denominator is 0 the term is counted as 0.
auto_valid approximates the official "invalid GT" exclusion (the public gt.csv has no invalid marker).
Usage as a module: evaluate(probs, gt, sources, threshold=0.5) -> dict (mean over datasets)."""
import numpy as np

def _metrics(prob, gt, valid=None, thr=0.5):
    pred = prob > thr
    g = gt > 0.5
    if valid is None:
        valid = np.ones(gt.shape[1], bool)
    # label-based, only attributes valid in this dataset
    tp = (pred & g).sum(0); fn = (~pred & g).sum(0)
    tn = (~pred & ~g).sum(0); fp = (pred & ~g).sum(0)
    pos = tp / np.maximum(tp + fn, 1); neg = tn / np.maximum(tn + fp, 1)
    ma = ((pos + neg) / 2)[valid].mean()
    f1 = (2 * tp / np.maximum(2 * tp + fp + fn, 1))[valid].mean()
    # instance-based over valid attributes
    P = pred[:, valid]; G = g[:, valid]
    inter = (P & G).sum(1); union = (P | G).sum(1)
    acc = np.where(union > 0, inter / np.maximum(union, 1), 0).mean()
    prec_i = np.where(P.sum(1) > 0, inter / np.maximum(P.sum(1), 1), 0)
    rec_i = np.where(G.sum(1) > 0, inter / np.maximum(G.sum(1), 1), 0)
    prec, rec = prec_i.mean(), rec_i.mean()
    f1i = 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0
    avg = 2 * ma * f1i / (ma + f1i) if ma + f1i > 0 else 0.0
    return dict(Challenge_Avg=avg, mA=ma, Label_F1=f1, Instance_Acc=acc,
                Instance_Prec=prec, Instance_Rec=rec, Instance_F1=f1i)

def evaluate(prob, gt, sources, thr=0.5, valid_by_source=None, auto_valid=True):
    """Each dataset is scored on its own, then the metrics are averaged over datasets."""
    prob, gt, sources = np.asarray(prob), np.asarray(gt), np.asarray(sources)
    per = {}
    for s in sorted(set(sources)):
        m = sources == s
        v = None if valid_by_source is None else valid_by_source.get(s)
        if v is None and auto_valid:
            v = gt[m].sum(0) > 0   # approximation: attributes never positive in a dataset ~ 'invalid' there
        per[s] = _metrics(prob[m], gt[m], v, thr)
    keys = next(iter(per.values())).keys()
    mean = {k: float(np.mean([per[s][k] for s in per])) for k in keys}
    return mean, per

In [ ]:
import sys, importlib
sys.path.insert(0, str(DRIVE / 'code'))
import upar_eval; importlib.reload(upar_eval)
from upar_eval import evaluate

# sanity checks of the metric code itself
rng = np.random.default_rng(0)
g_ = (rng.random((500, 40)) < 0.3).astype(np.uint8); s_ = np.array(['a', 'b'] * 250)
perfect, _ = evaluate(g_.astype(float), g_, s_)
assert all(abs(v - 1) < 1e-9 for v in perfect.values()), perfect
zero, _ = evaluate(np.zeros(g_.shape), g_, s_)
assert abs(zero['mA'] - 0.5) < 1e-9 and zero['Instance_F1'] == 0 and zero['Challenge_Avg'] == 0, zero
print('metric sanity checks passed\n')

prior = ytr.mean(0)
print('train prior predicts present only:', [n for n, p in zip(NAMES, prior) if p > 0.5])
BASE = {'train prior': np.tile(prior, (len(yva), 1)), 'all-zero': np.zeros(yva.shape)}
VARIANTS = {'auto_valid=True (drop attrs with no positive in that source)': True,
            'auto_valid=False (all 40 attributes)': False}

rows = []
for vname, av in VARIANTS.items():
    for bname, prob in BASE.items():
        mean, per = evaluate(prob, yva, src_va, auto_valid=av)
        for s in SOURCES:
            rows.append({'variant': vname, 'baseline': bname, 'source': s, **per[s]})
        rows.append({'variant': vname, 'baseline': bname, 'source': 'MEAN over sources', **mean})
bdf = pd.DataFrame(rows)
display(bdf.set_index(['variant', 'baseline', 'source']).round(4))
bdf.to_csv(FIG / 'baseline_scores_val.csv', index=False)

KNOWN = {'Challenge_Avg': 0.547, 'mA': 0.500, 'Instance_F1': 0.605}
for vname in VARIANTS:
    r = bdf[(bdf.variant == vname) & (bdf.baseline == 'train prior') & (bdf.source == 'MEAN over sources')].iloc[0]
    ok = all(abs(r[k] - v) <= 0.0015 for k, v in KNOWN.items())
    print(f'{vname}: Challenge_Avg {r.Challenge_Avg:.4f}  mA {r.mA:.4f}  Instance_F1 {r.Instance_F1:.4f}  ->',
          'MATCHES your 0.547 / 0.500 / 0.605' if ok else 'differs from 0.547 / 0.500 / 0.605')

### How Challenge_Avg behaves

The harmonic mean is pulled towards the weaker of mA and Instance_F1.

In [ ]:
def harmonic(a, b):
    return 2 * a * b / (a + b) if a + b > 0 else 0.0

vals = [0.5, 0.6, 0.7, 0.8, 0.9]
print('Challenge_Avg for a given mA (rows) and Instance_F1 (columns):')
display(pd.DataFrame([[harmonic(a, b) for b in vals] for a in vals],
                     index=[f'mA={a}' for a in vals], columns=[f'F1={b}' for b in vals]).round(3))
a, b = 0.70, 0.80
print(f'+0.05 on the weaker metric (mA 0.70->0.75): {harmonic(a + .05, b) - harmonic(a, b):+.4f}')
print(f'+0.05 on the stronger one (F1 0.80->0.85): {harmonic(a, b + .05) - harmonic(a, b):+.4f}')
print('check vs leaderboard row 1: harmonic(0.8098, 0.7945) =', round(harmonic(0.8098, 0.7945), 4), '(published 0.8021)')
print('check vs your baseline:      harmonic(0.5, 0.491)   =', round(harmonic(0.5, 0.491), 4), '(published 0.4955)')
print('figures saved in', FIG, ':', sorted(p.name for p in FIG.glob('*')))

## Part 3. Image cache and GPU data pipeline

Every crop is resized once to 192x96 (aspect ratio kept, padded) and stored as uint8, so training never decodes images. Augmentation runs on the GPU.

### Setup

In [ ]:
import os, sys, json, time, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/upar')
CODE = DRIVE / 'code'; CODE.mkdir(parents=True, exist_ok=True)
DATA = Path('/content/data')
sys.path.insert(0, str(CODE))

for d in ('Market1501/bounding_box_train', 'PA100k/release_data/release_data', 'PETA/images', 'annotations/task1/train'):
    assert (DATA / d).exists(), f'{d} missing in {DATA}: run notebook 1 first (it restores the data from Drive).'
print('data present. CPU cores:', os.cpu_count())

### Resolution and size estimate

In [ ]:
H, W, MODE = 192, 96, 'pad'            # HxW of the cache; MODE 'pad' keeps aspect ratio, 'stretch' distorts
WORKERS = max(1, min(os.cpu_count() or 2, 8))
CACHE_DIR = Path(f'/content/cache/h{H}w{W}_{MODE}')
N = {'train': 97669, 'val': 33407}
DRIVE_FREE_GB = 6.0                    # ~7 GB minus the 0.84 GB of archives; update from drive.google.com/settings/storage
# NB: shutil.disk_usage('/content/drive/...') reports the VM disk, NOT your Drive quota.

print(f'{"cache size HxW":>14} {"train GB":>9} {"val GB":>7} {"total GB":>9}  fits Drive (keep 1 GB)?  fits RAM page cache (<=8 GB)?')
for h, w in [(128, 64), (160, 80), (192, 96), (224, 112), (256, 128)]:
    tr, va = N['train'] * h * w * 3 / 1e9, N['val'] * h * w * 3 / 1e9
    mark = '  <== selected' if (h, w) == (H, W) else ''
    print(f'{h:>9}x{w:<4} {tr:9.2f} {va:7.2f} {tr + va:9.2f}  {"yes" if tr + va + 1 <= DRIVE_FREE_GB else "no ":>22}  {"yes" if tr + va <= 8 else "no":>28}{mark}')
need = sum(N.values()) * H * W * 3 / 1e9
print(f'\nselected {H}x{W}: {need:.2f} GB; local disk free {shutil.disk_usage("/content").free / 1e9:.1f} GB')
assert need < shutil.disk_usage('/content').free / 1e9 - 5

### Preprocessing module

The same function is used at training time and inside `run.py`.

In [ ]:
%%writefile /content/drive/MyDrive/upar/code/upar_preproc.py
"""Shared preprocessing. The cache builder (training) and run.py (inference) MUST use this same function."""
import numpy as np
from PIL import Image

PAD_RGB = (124, 116, 104)   # ImageNet mean in uint8 -> ~0 after normalisation


def letterbox(img, H, W, mode="pad", pad_rgb=PAD_RGB):
    """PIL image -> uint8 array (H, W, 3).

    mode="pad":     keep aspect ratio, fit inside HxW, centre on a mean-coloured canvas.
    mode="stretch": plain resize to HxW (distorts aspect).
    """
    img = img.convert("RGB")
    w, h = img.size
    if mode == "stretch":
        return np.asarray(img.resize((W, H), Image.BICUBIC), dtype=np.uint8)
    s = min(W / w, H / h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    img = img.resize((nw, nh), Image.BICUBIC)
    if (nw, nh) == (W, H):
        return np.asarray(img, dtype=np.uint8)
    canvas = Image.new("RGB", (W, H), pad_rgb)
    canvas.paste(img, ((W - nw) // 2, (H - nh) // 2))
    return np.asarray(canvas, dtype=np.uint8)


def load_resized(path, H, W, mode="pad"):
    with Image.open(path) as im:
        return letterbox(im, H, W, mode)

### Resumable cache builder

In [ ]:
%%writefile /content/drive/MyDrive/upar/code/upar_cache.py
"""Resumable uint8 image cache: one .npy memmap per split, built in chunks."""
import json
import os
import time
from math import ceil
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from upar_preproc import load_resized

CHUNK = 512
_G = {}


def _init(root, npy, paths, H, W, mode):
    _G.update(root=Path(root), npy=npy, paths=paths, H=H, W=W, mode=mode)


def _work(cid):
    g = _G
    lo, hi = cid * CHUNK, min((cid + 1) * CHUNK, len(g["paths"]))
    mm = np.load(g["npy"], mmap_mode="r+")
    bad = []
    for i in range(lo, hi):
        try:
            mm[i] = load_resized(g["root"] / g["paths"][i], g["H"], g["W"], g["mode"])
        except Exception as e:
            mm[i] = 0
            bad.append((i, g["paths"][i], repr(e)))
    mm.flush()
    return cid, hi - lo, bad


def _atomic_write(path, text):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text)
    os.replace(tmp, path)


def build_split(split, data_dir, cache_dir, H, W, mode="pad", workers=2):
    """Build (or resume) <cache_dir>/<split>_images.npy of shape (N, H, W, 3) uint8.

    Also writes <split>_labels.npy (N, 40 uint8), <split>_index.csv (row, image, source),
    attributes.json, and <split>_done.json (finished chunk ids, updated after every chunk).
    """
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    gt = pd.read_csv(data_dir / "annotations" / "task1" / split / "gt.csv")
    paths = gt.iloc[:, 0].tolist()
    attrs = list(gt.columns[1:])
    N = len(paths)
    assert len(attrs) == 40, f"expected 40 attribute columns, got {len(attrs)}"

    pd.DataFrame({"row": range(N), "image": paths,
                  "source": [p.split("/")[0] for p in paths]}).to_csv(cache_dir / f"{split}_index.csv", index=False)
    np.save(cache_dir / f"{split}_labels.npy", gt.iloc[:, 1:].to_numpy(dtype=np.uint8))
    (cache_dir / "attributes.json").write_text(json.dumps(attrs))

    npy, state_f, bad_f = (cache_dir / f"{split}_images.npy", cache_dir / f"{split}_done.json",
                           cache_dir / f"{split}_bad.txt")
    meta = dict(H=H, W=W, mode=mode, N=N, chunk=CHUNK)
    done = set()
    state = json.loads(state_f.read_text()) if state_f.exists() else None
    if npy.exists() and state and state["meta"] == meta:
        done = set(state["done"])
    else:
        mm = np.lib.format.open_memmap(npy, mode="w+", dtype=np.uint8, shape=(N, H, W, 3))
        del mm
        bad_f.write_text("")
        if state_f.exists():
            state_f.unlink()

    n_chunks = ceil(N / CHUNK)
    todo = [c for c in range(n_chunks) if c not in done]
    print(f"{split}: {N} images, {len(done)}/{n_chunks} chunks already done, {len(todo)} to build")
    if todo:
        t0 = time.time()
        total = sum(min(CHUNK, N - c * CHUNK) for c in todo)
        with Pool(workers, initializer=_init, initargs=(data_dir, str(npy), paths, H, W, mode)) as pool, \
                tqdm(total=total, unit="img", desc=split) as bar:
            for cid, n, bad in pool.imap_unordered(_work, todo):
                done.add(cid)
                bar.update(n)
                if bad:
                    with open(bad_f, "a") as f:
                        f.writelines("\t".join(map(str, b)) + "\n" for b in bad)
                _atomic_write(state_f, json.dumps({"meta": meta, "done": sorted(done)}))
        print(f"{split}: built in {(time.time() - t0) / 60:.1f} min ({total / (time.time() - t0):.0f} img/s)")
    n_bad = sum(1 for _ in open(bad_f)) if bad_f.exists() else 0
    return {"split": split, "N": N, "chunks_done": len(done), "n_chunks": n_chunks, "bad_images": n_bad}

### Decode speed

In [ ]:
import importlib, time, numpy as np, pandas as pd
import upar_preproc; importlib.reload(upar_preproc)

gt = pd.read_csv(DATA / 'annotations/task1/train/gt.csv')
sample = gt.iloc[:, 0].sample(600, random_state=0).tolist()
t = time.time()
for p in sample:
    upar_preproc.load_resized(DATA / p, H, W, MODE)
per_img = (time.time() - t) / len(sample)
total = sum(N.values())
print(f'{per_img * 1000:.1f} ms/image on one core')
print(f'estimated build time with {WORKERS} workers: ~{total * per_img / WORKERS * 1.15 / 60:.1f} min for {total} images (estimate, +15% overhead)')

### Build the cache

Saves progress after every chunk, so an interrupted run just continues when the cell is run again.

In [ ]:
import upar_cache; importlib.reload(upar_cache)
t0 = time.time()
summary = [upar_cache.build_split(s, DATA, CACHE_DIR, H, W, MODE, workers=WORKERS) for s in ('train', 'val')]
print(json.dumps(summary, indent=1))
print(f'total {(time.time() - t0) / 60:.1f} min; cache on disk: {sum(f.stat().st_size for f in CACHE_DIR.glob("*.npy")) / 1e9:.2f} GB')
assert all(s['chunks_done'] == s['n_chunks'] and s['bad_images'] == 0 for s in summary), 'incomplete cache or unreadable images: see *_bad.txt'

### Check the cache

In [ ]:
rng = np.random.default_rng(0)
for split in ('train', 'val'):
    X = np.load(CACHE_DIR / f'{split}_images.npy', mmap_mode='r')
    idx = pd.read_csv(CACHE_DIR / f'{split}_index.csv')
    Y = np.load(CACHE_DIR / f'{split}_labels.npy')
    assert len(X) == len(idx) == len(Y) == N[split], (len(X), len(idx), len(Y))
    rows = rng.choice(len(idx), 40, replace=False)
    same = all(np.array_equal(X[r], upar_preproc.load_resized(DATA / idx.image[r], H, W, MODE)) for r in rows)
    blank = int(sum(X[r].max() == 0 for r in rows))
    print(f'{split}: images {X.shape} {X.dtype} | labels {Y.shape} max={Y.max()} | 40 random rows identical to a fresh decode: {same} | blank rows: {blank}')
    assert same and blank == 0
print('cache OK')

### Restore in a new session

Rebuilds only what is missing, after the data has been restored from the Drive archives.

In [ ]:
# ---- NEW SESSION RESTORE ----
import os, sys, json, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
sys.path.insert(0, '/content/drive/MyDrive/upar/code')
import upar_cache

H, W, MODE = 192, 96, 'pad'                      # must match the values in section 2
DATA, CACHE_DIR = Path('/content/data'), Path(f'/content/cache/h{H}w{W}_{MODE}')
WORKERS = max(1, min(os.cpu_count() or 2, 8))
for d in ('Market1501/bounding_box_train', 'PA100k/release_data/release_data', 'PETA/images'):
    assert (DATA / d).exists(), 'run notebook 1 first'
t0 = time.time()
print([upar_cache.build_split(s, DATA, CACHE_DIR, H, W, MODE, workers=WORKERS) for s in ('train', 'val')])
print(f'done in {(time.time() - t0) / 60:.1f} min')

### Loader and GPU augmentation

In [ ]:
%%writefile /content/drive/MyDrive/upar/code/upar_data.py
"""Fast training data path: uint8 memmap cache -> pinned batches -> GPU augmentation (no PIL in the loop)."""
import math
import queue
import random
import threading
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


class CacheSplit:
    def __init__(self, cache_dir, split):
        d = Path(cache_dir)
        self.images = np.load(d / f"{split}_images.npy", mmap_mode="r")   # N,H,W,3 uint8
        self.labels = np.load(d / f"{split}_labels.npy")                  # N,40 uint8
        assert len(self.images) == len(self.labels)

    def __len__(self):
        return len(self.images)


class Prefetcher:
    """Background thread builds shuffled uint8 batches (pinned memory); the main thread copies them to the GPU."""

    def __init__(self, split, batch_size=128, shuffle=True, drop_last=True, device="cuda", depth=4, seed=0):
        self.split, self.B, self.shuffle, self.drop_last = split, batch_size, shuffle, drop_last
        self.device, self.depth, self.seed, self.epoch = device, depth, seed, 0
        self.pin = str(device).startswith("cuda")

    def __len__(self):
        n = len(self.split)
        return n // self.B if self.drop_last else math.ceil(n / self.B)

    @staticmethod
    def _put(q, item, stop):
        while not stop.is_set():
            try:
                q.put(item, timeout=0.1)
                return
            except queue.Full:
                pass

    def _produce(self, q, order, stop):
        try:
            for b in range(len(self)):
                if stop.is_set():
                    return
                ids = np.sort(order[b * self.B:(b + 1) * self.B])      # sorted ids = friendlier memmap reads
                x = torch.from_numpy(np.ascontiguousarray(self.split.images[ids]))
                y = torch.from_numpy(self.split.labels[ids])
                if self.pin:
                    x, y = x.pin_memory(), y.pin_memory()
                self._put(q, (x, y), stop)
            self._put(q, None, stop)
        except Exception as e:                                          # surface errors in the main thread
            self._put(q, e, stop)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        order = rng.permutation(len(self.split)) if self.shuffle else np.arange(len(self.split))
        q, stop = queue.Queue(maxsize=self.depth), threading.Event()
        threading.Thread(target=self._produce, args=(q, order, stop), daemon=True).start()
        try:
            while True:
                item = q.get()
                if item is None:
                    return
                if isinstance(item, Exception):
                    raise item
                x, y = item
                yield x.to(self.device, non_blocking=True), y.to(self.device, non_blocking=True).float()
        finally:
            stop.set()


class GPUAugment:
    """uint8 (B,H,W,3) on GPU -> normalised float (B,3,H,W). All ops are batched torch ops.

    Deliberately NO hue shift, channel shuffle or random grayscale: 24 of the 40 attributes are colours.
    Horizontal flip is safe: no attribute is left/right specific.
    """

    def __init__(self, device="cuda", flip=0.5, affine=0.8, rot_deg=5.0, zoom=(0.9, 1.1), shift=0.05,
                 photo=0.8, bright=0.25, contrast=0.25, sat=0.25, lowres=0.2, erase=0.25, channels_last=True):
        self.device, self.flip, self.affine, self.rot, self.zoom, self.shift = device, flip, affine, rot_deg, zoom, shift
        self.photo, self.bright, self.contrast, self.sat = photo, bright, contrast, sat
        self.lowres, self.erase, self.channels_last = lowres, erase, channels_last
        self.mean = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)
        self.std = torch.tensor(STD, device=device).view(1, 3, 1, 1)

    def _u(self, B, lo, hi):
        return torch.empty(B, device=self.device).uniform_(lo, hi)

    def _gate(self, B, p):
        return (torch.rand(B, device=self.device) < p).view(B, 1, 1, 1)

    def _finish(self, x):
        return x.contiguous(memory_format=torch.channels_last) if self.channels_last else x.contiguous()

    @torch.no_grad()
    def eval(self, xb):
        x = xb.permute(0, 3, 1, 2).float().div_(255)
        return self._finish((x - self.mean) / self.std)

    @torch.no_grad()
    def train(self, xb):
        B, H, W, _ = xb.shape
        dev = self.device
        x = xb.permute(0, 3, 1, 2).float().div_(255)

        # photometric: brightness, contrast, saturation (each sample on/off with prob `photo`)
        on = self._gate(B, self.photo).float()
        x = x * (1 + on * self._u(B, -self.bright, self.bright).view(B, 1, 1, 1))
        m = x.mean((1, 2, 3), keepdim=True)
        x = (x - m) * (1 + on * self._u(B, -self.contrast, self.contrast).view(B, 1, 1, 1)) + m
        g = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        x = ((x - g) * (1 + on * self._u(B, -self.sat, self.sat).view(B, 1, 1, 1)) + g).clamp_(0, 1)

        # low-resolution simulation (domain shift): down-up sample a random subset, one scale per batch
        if self.lowres > 0:
            s = random.uniform(0.4, 0.8)
            small = F.interpolate(x, scale_factor=s, mode="bilinear", antialias=True,
                                  align_corners=False, recompute_scale_factor=True)
            low = F.interpolate(small, size=(H, W), mode="bilinear", align_corners=False)
            x = torch.where(self._gate(B, self.lowres), low, x)

        x = (x - self.mean) / self.std

        if self.flip > 0:
            x = torch.where(self._gate(B, self.flip), x.flip(3), x)

        # small rotation / zoom / shift in one grid_sample; zeros after normalisation == mean colour == pad colour
        if self.affine > 0:
            a = (torch.rand(B, device=dev) < self.affine).float()
            ang = self._u(B, -self.rot, self.rot) * math.pi / 180 * a
            z = 1 + (self._u(B, self.zoom[0], self.zoom[1]) - 1) * a
            tx, ty = self._u(B, -self.shift, self.shift) * a, self._u(B, -self.shift, self.shift) * a
            c, s_ = torch.cos(ang), torch.sin(ang)
            theta = torch.zeros(B, 2, 3, device=dev)
            theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = c / z, -s_ * (H / W) / z, tx
            theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = s_ * (W / H) / z, c / z, ty
            grid = F.affine_grid(theta, list(x.shape), align_corners=False)
            x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)

        # random erasing (occlusion), filled with the mean colour (0 after normalisation)
        if self.erase > 0:
            area = self._u(B, 0.02, 0.15) * H * W
            r = torch.exp(self._u(B, math.log(0.3), math.log(3.3)))       # r = h / w
            eh, ew = torch.sqrt(area * r).clamp(max=H - 1), torch.sqrt(area / r).clamp(max=W - 1)
            top, left = torch.rand(B, device=dev) * (H - eh), torch.rand(B, device=dev) * (W - ew)
            ys = torch.arange(H, device=dev).view(1, H, 1)
            xs = torch.arange(W, device=dev).view(1, 1, W)
            box = ((ys >= top.view(B, 1, 1)) & (ys < (top + eh).view(B, 1, 1)) &
                   (xs >= left.view(B, 1, 1)) & (xs < (left + ew).view(B, 1, 1)))
            box = box & self._gate(B, self.erase).view(B, 1, 1)
            x = x.masked_fill(box.unsqueeze(1), 0.0)

        return self._finish(x)

### Look at the data

Top row: cached images. Bottom row: the same images after augmentation.

In [ ]:
import torch, matplotlib.pyplot as plt, upar_data
importlib.reload(upar_data)
from upar_data import CacheSplit, Prefetcher, GPUAugment

train = CacheSplit(CACHE_DIR, 'train')
aug = GPUAugment()
x, y = next(iter(Prefetcher(train, batch_size=16)))
xa = aug.train(x)
mean, std = aug.mean, aug.std
show = lambda t: (t * std + mean).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()
raw = x.cpu().numpy()
fig, ax = plt.subplots(2, 12, figsize=(18, 6))
for i in range(12):
    ax[0, i].imshow(raw[i]); ax[1, i].imshow(show(xa)[i]); ax[0, i].axis('off'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

### Throughput benchmark

A randomly initialised ConvNeXt-Tiny, to see whether the data path or the GPU limits training.

In [ ]:
import torchvision, torch.nn as nn
B, dev = 128, 'cuda'
loader = lambda: Prefetcher(train, batch_size=B, device=dev)

def timed(fn, n_batches):
    fn(3)                                   # warm-up
    torch.cuda.synchronize(); t = time.time(); fn(n_batches); torch.cuda.synchronize()
    return n_batches * B / (time.time() - t)

def run_loader(n):
    for i, (x, y) in enumerate(loader()):
        if i + 1 >= n: break

xb, yb = next(iter(loader()))
def run_aug(n):
    for _ in range(n): aug.train(xb)

model = torchvision.models.convnext_tiny(weights=None, num_classes=40).to(dev).to(memory_format=torch.channels_last)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler('cuda')
lossf = nn.BCEWithLogitsLoss()
def step(x, y):
    with torch.autocast('cuda', dtype=torch.float16):
        loss = lossf(model(x).float(), y)
    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()

xs = aug.train(xb)
def run_model(n):
    for _ in range(n): step(xs, yb)

def run_full(n):
    for i, (x, y) in enumerate(loader()):
        step(aug.train(x), y)
        if i + 1 >= n: break

res = {}
for name, fn, n in [('(a) loader only: memmap -> pinned -> GPU', run_loader, 40),
                    ('(b) GPU augmentation only', run_aug, 40),
                    ('(c) model fwd+bwd only (synthetic batch, fp16)', run_model, 20),
                    ('(d) full: loader + augmentation + train step', run_full, 30)]:
    res[name] = timed(fn, n); print(f'{name:55s} {res[name]:8.0f} img/s')

full = res['(d) full: loader + augmentation + train step']; model_only = res['(c) model fwd+bwd only (synthetic batch, fp16)']
print(f'\nfull loop is {100 * full / model_only:.0f}% of model-only speed ->',
      'GPU-bound (data pipeline is fine)' if full > 0.85 * model_only else 'data pipeline is slowing training: send me these numbers')
print(f'estimated training time per epoch on {len(train)} train images: {len(train) / full / 60:.1f} min (ConvNeXt-Tiny, {H}x{W}, batch {B}, excluding validation)')
print(f'GPU memory peak: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')

## Part 4. Training

The script supports ConvNeXt-Tiny, ResNet-50, CLIP ViT-B/16 and CLIP ViT-L/14, leave-one-source-out runs (`--holdout`) and a train+val mode (`--final`). Every run saves a checkpoint each epoch and resumes by itself.

### CLIP image tower

A plain-PyTorch copy of the CLIP ViT image tower (B/16 and L/14), so the submission needs neither `open_clip` nor `timm`. The weights are fetched with `open_clip` at training time only.

In [ ]:
%%writefile /content/drive/MyDrive/upar/code/upar_clipvit.py
"""Plain-PyTorch CLIP ViT image towers (same parameter names as open_clip) + a 40-way PAR head.

Inference needs only torch. open_clip is used only at training time, to fetch the pretrained weights.
The model takes the normalised 192x96 crops and resizes them to its own input size
(224x112 for ViT-B/16, 196x98 for ViT-L/14 by default).
"""
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

SPECS = {
    'clip_b16': dict(model='ViT-B-16', patch=16, width=768, layers=12, heads=12, out_dim=512, in_size=(224, 112)),
    'clip_l14': dict(model='ViT-L-14', patch=14, width=1024, layers=24, heads=16, out_dim=768, in_size=(196, 98)),
}
DIM_KEYS = ('patch', 'width', 'layers', 'heads', 'out_dim')


class QuickGELU(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(1.702 * x)


class ResBlock(nn.Module):
    def __init__(self, d, heads, act):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ln_2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(OrderedDict([('c_fc', nn.Linear(d, 4 * d)),
                                              ('gelu', QuickGELU() if act == 'quickgelu' else nn.GELU()),
                                              ('c_proj', nn.Linear(4 * d, d))]))

    def forward(self, x):
        y = self.ln_1(x)
        x = x + self.attn(y, y, y, need_weights=False)[0]
        return x + self.mlp(self.ln_2(x))


class Blocks(nn.Module):
    def __init__(self, d, layers, heads, act):
        super().__init__()
        self.resblocks = nn.ModuleList([ResBlock(d, heads, act) for _ in range(layers)])


class ClipViT(nn.Module):
    def __init__(self, grid=(14, 7), patch=16, width=768, layers=12, heads=12, out_dim=512, act='quickgelu'):
        super().__init__()
        self.grid = tuple(grid)
        self.conv1 = nn.Conv2d(3, width, patch, patch, bias=False)
        s = width ** -0.5
        self.class_embedding = nn.Parameter(s * torch.randn(width))
        self.positional_embedding = nn.Parameter(s * torch.randn(grid[0] * grid[1] + 1, width))
        self.ln_pre = nn.LayerNorm(width)
        self.transformer = Blocks(width, layers, heads, act)
        self.ln_post = nn.LayerNorm(width)
        self.proj = nn.Parameter(s * torch.randn(width, out_dim))

    def forward(self, x):
        x = self.conv1(x).flatten(2).transpose(1, 2)
        cls = self.class_embedding.to(x.dtype).expand(x.shape[0], 1, -1)
        x = torch.cat([cls, x], dim=1) + self.positional_embedding.to(x.dtype)
        x = self.ln_pre(x)
        for blk in self.transformer.resblocks:
            x = blk(x)
        return self.ln_post(x[:, 0]) @ self.proj


class ClipPAR(nn.Module):
    def __init__(self, in_size=(224, 112), act='quickgelu', n_out=40, patch=16, width=768, layers=12, heads=12,
                 out_dim=512, arch='clip_b16'):
        super().__init__()
        self.in_size = tuple(in_size)
        self.visual = ClipViT(grid=(self.in_size[0] // patch, self.in_size[1] // patch), patch=patch, width=width,
                              layers=layers, heads=heads, out_dim=out_dim, act=act)
        self.head = nn.Linear(out_dim, n_out)
        self.export_cfg = {'arch': arch, 'in_size': list(self.in_size), 'act': act, 'mean': list(CLIP_MEAN),
                           'std': list(CLIP_STD), 'patch': patch, 'width': width, 'layers': layers, 'heads': heads,
                           'out_dim': out_dim}

    def forward(self, x):
        if tuple(x.shape[-2:]) != self.in_size:
            x = F.interpolate(x, size=self.in_size, mode='bilinear', align_corners=False)
        return self.head(self.visual(x))


def resize_pos_embed(pe, grid):
    cls, g = pe[:1], pe[1:]
    n = int(round(g.shape[0] ** 0.5))
    g = g.reshape(1, n, n, -1).permute(0, 3, 1, 2).float()
    g = F.interpolate(g, size=tuple(grid), mode='bicubic', align_corners=False, antialias=True)
    return torch.cat([cls, g.permute(0, 2, 3, 1).reshape(grid[0] * grid[1], -1).to(pe.dtype)], 0)


def load_pretrained_visual(model='ViT-B-16', pretrained='openai'):
    import open_clip
    err = None
    for name in (model + '-quickgelu', model):
        try:
            vis = open_clip.create_model(name, pretrained=pretrained).visual.eval()
            act = 'quickgelu' if 'QuickGELU' in type(vis.transformer.resblocks[0].mlp.gelu).__name__ else 'gelu'
            return vis, act
        except Exception as e:
            err = e
    raise RuntimeError(f'could not load {model} ({pretrained}) weights: {err!r}')


def load_openai_visual():                      # kept for older code
    return load_pretrained_visual('ViT-B-16', 'openai')


def build_clip_par(pretrained=True, spec='clip_b16', in_size=None, act='quickgelu', tag='openai'):
    sp = SPECS[spec]
    in_size = tuple(in_size) if in_size else sp['in_size']
    assert all(v % sp['patch'] == 0 for v in in_size), f'{in_size} must be a multiple of the patch size {sp["patch"]}'
    dims = {k: sp[k] for k in DIM_KEYS}
    if not pretrained:
        return ClipPAR(in_size, act, 40, arch=spec, **dims)
    vis, act = load_pretrained_visual(sp['model'], tag)
    model = ClipPAR(in_size, act, 40, arch=spec, **dims)
    sd = {k: v.detach().float() for k, v in vis.state_dict().items()}
    del vis
    sd['positional_embedding'] = resize_pos_embed(sd['positional_embedding'], model.visual.grid)
    missing, unexpected = model.visual.load_state_dict(sd, strict=False)
    assert not missing, f'CLIP weights are missing keys: {missing}'
    if unexpected:
        print('ignored CLIP keys:', unexpected)
    return model


@torch.no_grad()
def check_against_open_clip(spec='clip_b16', device='cuda', tag='openai'):
    sp = SPECS[spec]
    vis, act = load_pretrained_visual(sp['model'], tag)
    g = 224 // sp['patch']
    ours = ClipViT(grid=(g, g), act=act, **{k: sp[k] for k in DIM_KEYS})
    missing, unexpected = ours.load_state_dict({k: v.float() for k, v in vis.state_dict().items()}, strict=False)
    assert not missing, missing
    vis, ours = vis.float().to(device).eval(), ours.to(device).eval()
    x = torch.randn(2, 3, 224, 224, device=device)
    a = vis(x)
    a = a[0] if isinstance(a, tuple) else a
    d = (a - ours(x)).abs().max().item()
    print(f'{spec}: activation={act} | max |open_clip - ours| = {d:.2e} | ignored keys: {unexpected}')
    assert d < 1e-3, 'ARCHITECTURE MISMATCH: do not train, send this output'
    del vis, ours
    torch.cuda.empty_cache()

### Training script

In [ ]:
%%writefile /content/drive/MyDrive/upar/code/upar_train.py
"""Resumable multi-label PAR training on the uint8 cache.

python upar_train.py --name NAME [--arch convnext_tiny|resnet50|clip_b16] [--epochs N] [--holdout PETA] [--final] [--speedtest]
Everything for a run lives in <runs-dir>/<name>/ on Drive: ckpt_last.pt (full state, resumed automatically),
best_state_dict.pt (plain weights of the best epoch), log.csv, val_logits_{best,last}.npy, meta.json.

Paths assume the Colab layout used in the project (/content/drive/MyDrive/upar/...).
"""
import argparse, csv, json, math, os, queue, random, sys, threading, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

sys.path.insert(0, '/content/drive/MyDrive/upar/code')
from upar_data import GPUAugment      # noqa: E402
from upar_eval import evaluate        # noqa: E402

SOURCES = ['Market1501', 'PA100k', 'PETA']
SRC_ID = {s: i for i, s in enumerate(SOURCES)}
CFG_KEYS = ['arch', 'epochs', 'batch_size', 'lr', 'balance', 'pw_cap', 'holdout', 'final', 'no_mask',
            'debug_steps', 'warmup_epochs', 'llrd', 'lr_head', 'clip_size', 'freeze_epochs']

def get_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument('--name', default='run')
    p.add_argument('--cache-dir', default='/content/cache/h192w96_pad')
    p.add_argument('--runs-dir', default='/content/drive/MyDrive/upar/runs')
    p.add_argument('--arch', default='convnext_tiny', choices=['convnext_tiny', 'resnet50', 'clip_b16', 'clip_l14'])
    p.add_argument('--llrd', type=float, default=0.75, help='layer-wise lr decay (clip_b16 only)')
    p.add_argument('--lr-head', type=float, default=1e-3, help='lr of the new head (CLIP only)')
    p.add_argument('--clip-size', default=None, help='CLIP input HxW, e.g. 196x98 (default: per model)')
    p.add_argument('--freeze-epochs', type=int, default=0, help='train only the head for the first N epochs (LP-FT)')
    p.add_argument('--epochs', type=int, default=20)
    p.add_argument('--batch-size', type=int, default=128)
    p.add_argument('--lr', type=float, default=2e-4)
    p.add_argument('--min-lr', type=float, default=1e-6)
    p.add_argument('--wd', type=float, default=0.05)
    p.add_argument('--warmup-epochs', type=float, default=1.0)
    p.add_argument('--grad-clip', type=float, default=5.0)
    p.add_argument('--balance', type=float, default=0.0,
                   help='source share ~ n_s**balance: 0 = equal share per source, 1 = natural (unbalanced)')
    p.add_argument('--pw-cap', type=float, default=10.0, help='cap of the per-attribute pos_weight')
    p.add_argument('--holdout', default=None, choices=SOURCES, help='leave-one-source-out: train without this source')
    p.add_argument('--final', action='store_true', help='train on train+val, no validation')
    p.add_argument('--no-mask', action='store_true', help='ablation: do not mask never-annotated attributes')
    p.add_argument('--speedtest', action='store_true')
    p.add_argument('--gpu-hours', type=float, default=1.5)
    p.add_argument('--debug-steps', type=int, default=0, help='if >0: steps per epoch (smoke tests only)')
    p.add_argument('--no-resume', action='store_true')
    p.add_argument('--seed', type=int, default=0)
    return p.parse_args(argv)

# ----------------------------------------------------------------------------- data
def load_split(cache_dir, split):
    d = Path(cache_dir)
    images = np.load(d / f'{split}_images.npy', mmap_mode='r')
    labels = np.load(d / f'{split}_labels.npy')
    idx = pd.read_csv(d / f'{split}_index.csv')
    assert idx['source'].isin(SRC_ID).all() and len(idx) == len(images) == len(labels)
    return images, labels, idx['source'].map(SRC_ID).to_numpy(np.int64), idx['source'].to_numpy()

class Parts:
    """Logical concatenation of (images_memmap, labels, src) triples."""

    def __init__(self, triples):
        self.triples = triples
        self.offsets = np.cumsum([0] + [len(t[0]) for t in triples])
        self.labels = np.concatenate([t[1] for t in triples])
        self.src = np.concatenate([t[2] for t in triples])

    def __len__(self):
        return int(self.offsets[-1])

    def gather(self, ids):
        ids = np.sort(np.asarray(ids))
        xs = []
        for k, (im, _, _) in enumerate(self.triples):
            lo, hi = self.offsets[k], self.offsets[k + 1]
            sel = ids[(ids >= lo) & (ids < hi)] - lo
            if len(sel):
                xs.append(im[sel])
        x = xs[0] if len(xs) == 1 else np.concatenate(xs)
        return x, self.labels[ids], self.src[ids]

class Loader:
    """Background thread: batches of uint8 images + labels + source ids in pinned memory -> GPU."""

    def __init__(self, parts, order, batch, device='cuda', drop_last=True, depth=4):
        self.parts, self.order, self.B = parts, np.asarray(order), batch
        self.device, self.drop_last, self.depth = device, drop_last, depth

    def __len__(self):
        n = len(self.order)
        return n // self.B if self.drop_last else math.ceil(n / self.B)

    @staticmethod
    def _put(q, item, stop):
        while not stop.is_set():
            try:
                q.put(item, timeout=0.1)
                return
            except queue.Full:
                pass

    def _produce(self, q, stop):
        try:
            for b in range(len(self)):
                if stop.is_set():
                    return
                x, y, s = self.parts.gather(self.order[b * self.B:(b + 1) * self.B])
                item = (torch.from_numpy(np.ascontiguousarray(x)).pin_memory(),
                        torch.from_numpy(np.ascontiguousarray(y)).pin_memory(),
                        torch.from_numpy(np.ascontiguousarray(s)).pin_memory())
                self._put(q, item, stop)
            self._put(q, None, stop)
        except Exception as e:
            self._put(q, e, stop)

    def __iter__(self):
        q, stop = queue.Queue(maxsize=self.depth), threading.Event()
        threading.Thread(target=self._produce, args=(q, stop), daemon=True).start()
        try:
            while True:
                item = q.get()
                if item is None:
                    return
                if isinstance(item, Exception):
                    raise item
                x, y, s = item
                yield (x.to(self.device, non_blocking=True), y.to(self.device, non_blocking=True).float(),
                       s.to(self.device, non_blocking=True))
        finally:
            stop.set()

def sampling(src, row_ok, balance):
    """Rows to draw from and their probabilities: share of source s ~ n_s ** balance."""
    rows = np.where(row_ok)[0]
    s = src[rows]
    n_s = np.bincount(s, minlength=3).astype(np.float64)
    w = np.zeros(len(rows))
    for k in range(3):
        if n_s[k] > 0:
            w[s == k] = n_s[k] ** (balance - 1.0)
    return rows, w / w.sum()

def source_masks(labels, src, use_mask):
    """(3, 40) float: 0 where the source never has a positive (never annotated), computed on TRAIN rows only."""
    M = np.ones((3, 40), np.float32)
    if use_mask:
        for k in range(3):
            sel = src == k
            if sel.any():
                M[k] = (labels[sel].sum(0) > 0).astype(np.float32)
    return M

# ----------------------------------------------------------------------------- model / optim
def build_model(arch, clip_size=None):
    if arch == 'convnext_tiny':
        m = torchvision.models.convnext_tiny(weights=torchvision.models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, 40)
        head = m.classifier[2]
    elif arch == 'resnet50':
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, 40)
        head = m.fc
    else:
        from upar_clipvit import build_clip_par
        in_size = tuple(int(v) for v in clip_size.split('x')) if clip_size else None
        m = build_clip_par(pretrained=True, spec=arch, in_size=in_size)
        head = m.head
    return m, head

def make_optimizer(model, lr, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        (no_decay if p.ndim <= 1 or n.endswith('.bias') or 'layer_scale' in n else decay).append(p)
    return torch.optim.AdamW([{'params': decay, 'weight_decay': wd},
                              {'params': no_decay, 'weight_decay': 0.0}], lr=lr)

def make_optimizer_clip(model, lr, lr_head, wd, llrd):
    # layer-wise lr decay for the CLIP tower (deeper blocks get a larger lr); the new head gets lr_head
    L = len(model.visual.transformer.resblocks)
    groups = {}
    for n, p in model.named_parameters():
        if n.startswith('head.'):
            lid = L + 2
        elif n.startswith('visual.transformer.resblocks.'):
            lid = int(n.split('.')[3]) + 1
        elif n.startswith(('visual.ln_post', 'visual.proj')):
            lid = L + 1
        else:
            lid = 0
        no_wd = p.ndim <= 1 or n.endswith('.bias') or 'embedding' in n
        if (lid, no_wd) not in groups:
            g_lr = lr_head if lid == L + 2 else lr * llrd ** (L + 1 - lid)
            groups[(lid, no_wd)] = {'params': [], 'lr': g_lr, 'weight_decay': 0.0 if no_wd else wd}
        groups[(lid, no_wd)]['params'].append(p)
    return torch.optim.AdamW(list(groups.values()), lr=lr)

def make_sched(opt, total, warm, min_ratio):
    def f(step):
        if step < warm:
            return (step + 1) / max(1, warm)
        p = (step - warm) / max(1, total - warm)
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

# ----------------------------------------------------------------------------- train / eval
def run_epoch(model, loader, aug, opt, sched, scaler, M_t, pw_t, clip, cb=None, log_every=100):
    model.train()
    tot = torch.zeros((), device='cuda')
    n = 0
    for i, (xb, yb, sb) in enumerate(loader):
        x = aug.train(xb)
        m = M_t[sb]
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(x)
        l = F.binary_cross_entropy_with_logits(logits.float(), yb, pos_weight=pw_t, reduction='none')
        loss = (l * m).sum() / m.sum().clamp(min=1.0)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(opt)
        scaler.update()
        sched.step()
        tot += loss.detach()
        n += 1
        if cb is not None:
            cb(i)
        if (i + 1) % log_every == 0:
            print(f'    step {i + 1}/{len(loader)}  loss {loss.item():.4f}  lr {opt.param_groups[0]["lr"]:.2e}', flush=True)
    return (tot / max(n, 1)).item(), n

@torch.no_grad()
def predict(model, parts, aug, bs=256, max_batches=None):
    model.eval()
    loader = Loader(parts, np.arange(len(parts)), bs, drop_last=False)
    out = []
    for i, (xb, _, _) in enumerate(loader):
        with torch.autocast('cuda', dtype=torch.float16):
            lg = model(aug.eval(xb))
        out.append(lg.float().cpu())
        if max_batches and i + 1 >= max_batches:
            break
    return torch.cat(out).numpy()

def atomic_save(obj, path):
    tmp = path.with_suffix(path.suffix + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, path)

def get_rng(sampler_rng):
    return {'py': random.getstate(), 'np': np.random.get_state(), 'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state(), 'sampler': sampler_rng.bit_generator.state}

def set_rng(st, sampler_rng):
    random.setstate(st['py'])
    np.random.set_state(st['np'])
    torch.set_rng_state(st['torch'])
    torch.cuda.set_rng_state(st['cuda'])
    sampler_rng.bit_generator.state = st['sampler']

def append_log(path, row):
    new = not path.exists() or path.stat().st_size == 0
    with open(path, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(row))
        if new:
            w.writeheader()
        w.writerow(row)

def rss_gb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1e9
    except Exception:
        return float('nan')

# ----------------------------------------------------------------------------- main
def main():
    args = get_args()
    assert torch.cuda.is_available(), 'no GPU'
    assert not (args.final and args.holdout), '--final and --holdout cannot be combined'
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    torch.backends.cudnn.benchmark = True
    run_dir = Path(args.runs_dir) / args.name
    run_dir.mkdir(parents=True, exist_ok=True)

    names = json.loads((Path(args.cache_dir) / 'attributes.json').read_text())
    tr, va = load_split(args.cache_dir, 'train'), load_split(args.cache_dir, 'val')
    parts_val = Parts([va[:3]])
    parts_train = Parts([tr[:3], va[:3]]) if args.final else Parts([tr[:3]])

    row_ok = np.ones(len(parts_train), bool)
    if args.holdout:
        row_ok &= parts_train.src != SRC_ID[args.holdout]
    rows, w = sampling(parts_train.src, row_ok, args.balance)
    M = source_masks(tr[1], tr[2], not args.no_mask)

    y_rows = parts_train.labels[rows].astype(np.float64)
    m_rows = M[parts_train.src[rows]].astype(np.float64)
    pos = (w[:, None] * m_rows * y_rows).sum(0)
    neg = (w[:, None] * m_rows * (1 - y_rows)).sum(0)
    pw = np.clip(neg / np.maximum(pos, 1e-12), 1.0, args.pw_cap)
    p0 = np.clip(pos / np.maximum(pos + neg, 1e-12), 1e-3, 1 - 1e-3)

    B = args.batch_size
    steps_per_epoch = args.debug_steps if args.debug_steps > 0 else len(rows) // B
    total_steps = args.epochs * steps_per_epoch
    warm = int(args.warmup_epochs * steps_per_epoch)

    shares = {SOURCES[k]: round(float(w[parts_train.src[rows] == k].sum()), 3) for k in range(3)}
    print(f'run {args.name} | arch {args.arch} | train rows {len(rows)} | steps/epoch {steps_per_epoch} | '
          f'epochs {args.epochs} | holdout {args.holdout} | final {args.final}')
    print('sampling share per source:', shares)
    for k in range(3):
        masked = [names[a] for a in range(40) if M[k, a] == 0]
        if masked:
            print(f'  loss masked for {SOURCES[k]}:', masked)
    print('pos_weight: min %.1f max %.1f, %d attributes at the cap %.0f' %
          (pw.min(), pw.max(), int((pw >= args.pw_cap - 1e-9).sum()), args.pw_cap), flush=True)

    model, head = build_model(args.arch, args.clip_size)
    with torch.no_grad():
        nn.init.normal_(head.weight, std=0.01)
        head.bias.copy_(torch.tensor(np.log(p0 / (1 - p0)), dtype=torch.float32))
    model = model.cuda().to(memory_format=torch.channels_last)
    opt = (make_optimizer_clip(model, args.lr, args.lr_head, args.wd, args.llrd) if args.arch.startswith('clip')
           else make_optimizer(model, args.lr, args.wd))
    sched = make_sched(opt, total_steps, warm, args.min_lr / args.lr)
    scaler = torch.amp.GradScaler('cuda')
    aug = GPUAugment(device='cuda')
    if args.arch.startswith('clip'):                     # CLIP's own normalisation
        from upar_clipvit import CLIP_MEAN, CLIP_STD
        aug.mean = torch.tensor(CLIP_MEAN, device='cuda').view(1, 3, 1, 1)
        aug.std = torch.tensor(CLIP_STD, device='cuda').view(1, 3, 1, 1)
    M_t = torch.tensor(M, device='cuda')
    pw_t = torch.tensor(pw, dtype=torch.float32, device='cuda')
    sampler_rng = np.random.default_rng(args.seed)

    # ------------------------------------------------------------------ speed test
    if args.speedtest:
        n_steps = 200
        order = sampler_rng.choice(rows, size=n_steps * B, replace=True, p=w)
        marks = {}

        def cb(i):
            if i == 19 or i == n_steps - 1:
                torch.cuda.synchronize()
                marks[i] = time.time()
        torch.cuda.reset_peak_memory_stats()
        run_epoch(model, Loader(parts_train, order, B), aug, opt, sched, scaler, M_t, pw_t, args.grad_clip, cb, log_every=50)
        sec_step = (marks[n_steps - 1] - marks[19]) / (n_steps - 1 - 19)
        peak = torch.cuda.max_memory_allocated() / 1e9
        predict(model, parts_val, aug, max_batches=2)
        t = time.time(); predict(model, parts_val, aug, max_batches=15); torch.cuda.synchronize()
        eval_ips = 15 * 256 / (time.time() - t)
        t = time.time()
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict()}, run_dir / '_speedtest.pt')
        save_s = time.time() - t
        (run_dir / '_speedtest.pt').unlink()
        train_s = steps_per_epoch * sec_step
        eval_s = 0.0 if args.final else len(parts_val) / eval_ips
        per_epoch = train_s + eval_s + save_s
        print('\n================ SPEED TEST ================')
        print(f'{sec_step * 1000:.0f} ms/step = {B / sec_step:.0f} img/s | peak GPU memory {peak:.1f} GB (limit 15) | RSS {rss_gb():.1f} GB')
        print(f'seconds per epoch: train {train_s:.0f} + validation {eval_s:.0f} + checkpoint {save_s:.0f} = {per_epoch:.0f} s ({per_epoch / 60:.1f} min)')
        print('epochs that fit in a GPU-hour budget (90% of the budget is used, 10% kept as reserve):')
        for h in sorted({1.0, 1.5, 2.0, 3.0, args.gpu_hours}):
            print(f'   {h:>4.1f} h -> {int(h * 3600 * 0.9 // per_epoch)} epochs')
        return

    # ------------------------------------------------------------------ resume
    ck_path, log_path = run_dir / 'ckpt_last.pt', run_dir / 'log.csv'
    cfg = {k: getattr(args, k) for k in CFG_KEYS}
    start_epoch, best = 0, {'score': -1.0, 'epoch': 0}
    if ck_path.exists() and not args.no_resume:
        ck = torch.load(ck_path, map_location='cpu', weights_only=False)
        diff = {k: (ck['cfg'].get(k), cfg[k]) for k in CFG_KEYS if ck['cfg'].get(k) != cfg[k]}
        if diff:
            sys.exit(f'REFUSING to resume: settings differ from the checkpoint (old, new): {diff}\n'
                     f'Use a new --name, or --no-resume to start over in this folder.')
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
        start_epoch, best = ck['epoch'], ck['best']
        set_rng(ck['rng'], sampler_rng)
        if log_path.exists():
            df = pd.read_csv(log_path)
            df[df['epoch'] <= start_epoch].to_csv(log_path, index=False)
        print(f'RESUMED from {ck_path} after epoch {start_epoch}/{args.epochs} (best so far {best})', flush=True)
        del ck
    elif log_path.exists() and args.no_resume:
        log_path.unlink()
    if start_epoch >= args.epochs:
        print('run already finished; nothing to do'); return

    (run_dir / 'meta.json').write_text(json.dumps({
        'cfg': cfg, 'attribute_names': names, 'sources': SOURCES, 'loss_mask': M.tolist(), 'pos_weight': pw.tolist(),
        'sampling_share': shares, 'model_cfg': getattr(model, 'export_cfg', {}),
        'note': 'state_dict keys are ' + args.arch + ' with a 40-way head'}, indent=1))

    # ------------------------------------------------------------------ epochs
    for epoch in range(start_epoch, args.epochs):
        t0 = time.time()
        torch.cuda.reset_peak_memory_stats()
        frozen = epoch < args.freeze_epochs              # LP-FT: head only for the first epochs
        for n_, p_ in model.named_parameters():
            if not n_.startswith('head.'):
                p_.requires_grad_(not frozen)
        if frozen:
            print(f'epoch {epoch + 1}: backbone frozen, training the head only', flush=True)
        order = sampler_rng.choice(rows, size=steps_per_epoch * B, replace=True, p=w)
        loss, _ = run_epoch(model, Loader(parts_train, order, B), aug, opt, sched, scaler, M_t, pw_t, args.grad_clip)
        torch.cuda.synchronize()
        t_train = time.time() - t0

        row = {'epoch': epoch + 1, 'lr': opt.param_groups[0]['lr'], 'train_loss': loss, 'sec_train': t_train}
        improved = False
        if not args.final:
            t1 = time.time()
            logits = predict(model, parts_val, aug)
            mean, per = evaluate(1 / (1 + np.exp(-logits)), parts_val.labels, va[3])
            row['sec_eval'] = time.time() - t1
            for k in ('Challenge_Avg', 'mA', 'Instance_F1', 'Label_F1'):
                row[k] = mean[k]
            for s in SOURCES:
                for k in ('Challenge_Avg', 'mA', 'Instance_F1'):
                    row[f'{s}_{k}'] = per[s][k]
            score = per[args.holdout]['Challenge_Avg'] if args.holdout else mean['Challenge_Avg']
            row['select_score'] = score
            np.save(run_dir / 'val_logits_last.npy', logits.astype(np.float32))
            if score > best['score']:
                improved = True
                best = {'score': float(score), 'epoch': epoch + 1}
                atomic_save({k: v.detach().cpu() for k, v in model.state_dict().items()}, run_dir / 'best_state_dict.pt')
                np.save(run_dir / 'val_logits_best.npy', logits.astype(np.float32))
        peak = torch.cuda.max_memory_allocated() / 1e9
        row.update({'best_score': best['score'], 'best_epoch': best['epoch'], 'peak_gpu_gb': peak, 'rss_gb': rss_gb()})
        append_log(log_path, row)

        t2 = time.time()
        atomic_save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                     'scaler': scaler.state_dict(), 'epoch': epoch + 1, 'best': best, 'cfg': cfg,
                     'rng': get_rng(sampler_rng)}, ck_path)
        msg = f'epoch {epoch + 1}/{args.epochs} | loss {loss:.4f} | train {t_train:.0f}s'
        if not args.final:
            msg += (f' | val {row["sec_eval"]:.0f}s | Challenge_Avg {mean["Challenge_Avg"]:.4f} mA {mean["mA"]:.4f} '
                    f'F1 {mean["Instance_F1"]:.4f} | per source ' +
                    ' '.join(f'{s[:3]} {per[s]["Challenge_Avg"]:.3f}' for s in SOURCES) +
                    f' | select {row["select_score"]:.4f}' + (' *best*' if improved else ''))
        print(msg + f' | ckpt {time.time() - t2:.0f}s | peak GPU {peak:.1f} GB', flush=True)
        if peak > 14.5:
            print('WARNING: GPU memory close to the 15 GB limit; lower --batch-size')

    atomic_save({k: v.detach().cpu() for k, v in model.state_dict().items()}, run_dir / 'final_state_dict.pt')
    print(f'DONE. best epoch {best["epoch"]} (score {best["score"]:.4f}); weights: best_state_dict.pt / final_state_dict.pt')

if __name__ == '__main__':
    main()

### Pre-flight check

In [ ]:
import subprocess
from pathlib import Path
CACHE = Path('/content/cache/h192w96_pad')
for s in ('train', 'val'):
    assert (CACHE / f'{s}_images.npy').exists() and (CACHE / f'{s}_done.json').exists(), \
        'cache missing: run the "NEW SESSION RESTORE" cell of the cache section first'
assert (Path('/content/drive/MyDrive/upar/code') / 'upar_data.py').exists() and \
       (Path('/content/drive/MyDrive/upar/code') / 'upar_eval.py').exists(), 'upar_data.py / upar_eval.py missing on Drive'
print(subprocess.run(['python', '-m', 'py_compile', '/content/drive/MyDrive/upar/code/upar_train.py'],
                     capture_output=True, text=True))
print('preflight OK')

Commands we ran, one Colab session per run (`upar_train.py` is in `/content/drive/MyDrive/upar/code/`):

```
# held-out source experiments (train on two sources, score the third)
python -u upar_train.py --name ho_peta_base8   --holdout PETA --epochs 8
python -u upar_train.py --name ho_peta_bal05   --holdout PETA --balance 0.5 --epochs 8
python -u upar_train.py --name ho_peta_bal10   --holdout PETA --balance 1.0 --epochs 8
python -u upar_train.py --name ho_peta_nomask  --holdout PETA --no-mask --epochs 8
python -u upar_train.py --name ho_peta_r50_bal05 --holdout PETA --arch resnet50 --balance 0.5 --epochs 8
python -u upar_train.py --name ho_peta_clip_a  --holdout PETA --arch clip_b16 --balance 0.5 --epochs 5 --lr 3e-5 --llrd 0.75 --lr-head 1e-3
python -u upar_train.py --name ho_peta_clip_b  --holdout PETA --arch clip_b16 --balance 0.5 --epochs 5 --lr 1e-5 --llrd 0.7  --lr-head 1e-3
# (the same three balance settings with --holdout Market1501 for ConvNeXt-Tiny)

# final models, trained on train + validation (the three CLIP models of the submission)
python -u upar_train.py --name final_clip_b    --final --arch clip_b16 --balance 0.5 --epochs 5 --lr 1e-5 --llrd 0.7  --lr-head 1e-3
python -u upar_train.py --name final_clip_a    --final --arch clip_b16 --balance 0.5 --epochs 5 --lr 3e-5 --llrd 0.75 --lr-head 1e-3
python -u upar_train.py --name final_clip_b_s1 --final --arch clip_b16 --balance 0.5 --epochs 5 --lr 1e-5 --llrd 0.7  --lr-head 1e-3 --seed 1
```

The next two cells show the speed test and one of the final runs as we ran them.

### Speed test (ConvNeXt-Tiny)

In [ ]:
!python -u /content/drive/MyDrive/upar/code/upar_train.py --speedtest --name speedtest --arch convnext_tiny --gpu-hours 2.5

### CLIP: install and check against `open_clip`

`open_clip` is only needed during training, to download the pretrained weights. The check must show a tiny difference before any CLIP run starts.

In [ ]:
!pip install -q open_clip_torch
import open_clip; print('open_clip', open_clip.__version__)

In [ ]:
import sys, importlib, torch
sys.path.insert(0, '/content/drive/MyDrive/upar/code')
import upar_clipvit; importlib.reload(upar_clipvit)
upar_clipvit.check_against_open_clip()
m = upar_clipvit.build_clip_par(pretrained=True).cuda().eval()
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    print('PAR forward at 192x96 ->', tuple(m(torch.randn(2, 3, 192, 96, device='cuda')).shape))   # expect (2, 40)
del m; torch.cuda.empty_cache()

### One of the final runs

The other two final models were trained the same way by teammates on their own accounts.

In [ ]:
!python -u /content/drive/MyDrive/upar/code/upar_train.py --name final_clip_b --final --arch clip_b16 --balance 0.5 --epochs 5 --lr 1e-5 --llrd 0.7 --lr-head 1e-3

### CLIP ViT-L/14 (final improvement)

The larger tower is about 3x slower to train, so each model got 3 to 4 epochs on a T4 (a bit under 40 minutes per epoch at 196x98, batch 48, 12.9 GB peak). The input is 196x98 (a 14x7 grid of 14-pixel patches) instead of 224x112 so one epoch stays affordable. Both models were trained by teammates on their own Colab accounts; the weights were copied to `upar/runs/` on Drive.

```
# plain fine-tuning, 3 epochs
python -u upar_train.py --name final_l14_a    --final --arch clip_l14 --clip-size 196x98 --batch-size 48 --balance 0.5 --epochs 3 --warmup-epochs 0.5 --lr 1e-5 --llrd 0.85 --lr-head 1e-3
# linear probe then fine-tune: the first epoch trains only the head, 4 epochs in total
python -u upar_train.py --name final_l14_lpft --final --arch clip_l14 --clip-size 196x98 --batch-size 48 --balance 0.5 --epochs 4 --warmup-epochs 0.5 --lr 1e-5 --llrd 0.85 --lr-head 1e-3 --freeze-epochs 1
```

The `lr` shown in the logs (1.72e-07 at the peak) is the lowest layer group: 1e-5 x 0.85^25.

First the same check as for B/16: the plain-PyTorch L/14 must match `open_clip` before any run starts.

In [ ]:
!pip install -q open_clip_torch
import sys, importlib, torch
sys.path.insert(0, '/content/drive/MyDrive/upar/code')
import upar_clipvit; importlib.reload(upar_clipvit)
upar_clipvit.check_against_open_clip('clip_l14')
m = upar_clipvit.build_clip_par(pretrained=True, spec='clip_l14', in_size=(196, 98)).cuda().eval()
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    print('PAR forward at 196x98 ->', tuple(m(torch.randn(2, 3, 196, 98, device='cuda')).shape))   # expect (2, 40)
del m; torch.cuda.empty_cache()

Speed test before committing the session time (it reports seconds per epoch and peak memory):

In [ ]:
!python -u /content/drive/MyDrive/upar/code/upar_train.py --speedtest --name speedtest --final --arch clip_l14 --clip-size 196x98 --batch-size 48 --balance 0.5 --gpu-hours 1.6 --runs-dir /content/runs

The two final runs (about 2 h and 2.4 h of T4 time):

In [ ]:
!python -u /content/drive/MyDrive/upar/code/upar_train.py --name final_l14_a --final --arch clip_l14 --clip-size 196x98 --batch-size 48 --balance 0.5 --epochs 3 --warmup-epochs 0.5 --lr 1e-5 --llrd 0.85 --lr-head 1e-3 --runs-dir /content/runs

In [ ]:
!python -u /content/drive/MyDrive/upar/code/upar_train.py --name final_l14_lpft --final --arch clip_l14 --clip-size 196x98 --batch-size 48 --balance 0.5 --epochs 4 --warmup-epochs 0.5 --lr 1e-5 --llrd 0.85 --lr-head 1e-3 --freeze-epochs 1 --runs-dir /content/runs

## Part 5. Experiments

Main finding: on held-out PETA (Challenge_Avg, last epoch) ConvNeXt-Tiny reached 0.7384 to 0.7474 depending on the
sampling setting, ResNet-50 0.7342, CLIP ViT-B/16 0.7857 and 0.7876, and two CLIP models averaged 0.7912. The full
table is in `results/heldout_ablations.csv`. The cells below are the checks that did not change the final model.

### Does averaging models help? ConvNeXt-Tiny runs

Held-out logits of three PETA runs and two Market1501 runs, averaged.

In [ ]:
import sys, numpy as np, pandas as pd
from pathlib import Path
sys.path.insert(0, '/content/drive/MyDrive/upar/code')
from upar_eval import evaluate
RUNS = Path('/content/drive/MyDrive/upar/runs')
va = pd.read_csv('/content/data/annotations/task1/val/gt.csv')
Y = va.iloc[:, 1:].to_numpy(np.uint8); src = va.iloc[:, 0].str.split('/').str[0].to_numpy()

def sc(z, mask):
    m, _ = evaluate(1 / (1 + np.exp(-z[mask])), Y[mask], src[mask])
    return m['Challenge_Avg'], m['mA'], m['Instance_F1']

groups = {'PETA': (['ho_peta_base8', 'ho_peta_bal05', 'ho_peta_bal10'], src == 'PETA'),
          'Market1501': (['ho_market_base8', 'ho_market_bal05'], src == 'Market1501')}
for held, (runs, mask) in groups.items():
    Z = {r: np.load(RUNS / r / 'val_logits_last.npy') for r in runs if (RUNS / r / 'val_logits_last.npy').exists()}
    print(f'\n=== held-out source {held} (last-epoch logits) ===')
    for r, z in Z.items():
        print(f'{r:18s} Challenge_Avg {sc(z, mask)[0]:.4f}  mA {sc(z, mask)[1]:.4f}  F1 {sc(z, mask)[2]:.4f}')
    ens = np.mean(list(Z.values()), axis=0)
    e = sc(ens, mask); best = max(sc(z, mask)[0] for z in Z.values())
    print(f'ENSEMBLE of {len(Z)}      Challenge_Avg {e[0]:.4f}  mA {e[1]:.4f}  F1 {e[2]:.4f}   gain vs best member: {e[0] - best:+.4f}')

### Does averaging help? CLIP runs

The two CLIP runs, and CLIP mixed with ConvNeXt. Mixing ConvNeXt in made it worse, so the final ensemble is CLIP only.

In [ ]:
import sys, numpy as np, pandas as pd
from pathlib import Path
sys.path.insert(0, '/content/drive/MyDrive/upar/code')
from upar_eval import evaluate
RUNS = Path('/content/drive/MyDrive/upar/runs')
va = pd.read_csv('/content/data/annotations/task1/val/gt.csv')
Y = va.iloc[:, 1:].to_numpy(np.uint8); src = va.iloc[:, 0].str.split('/').str[0].to_numpy(); m = src == 'PETA'
Z = {r: np.load(RUNS / r / 'val_logits_last.npy') for r in ('ho_peta_clip_a', 'ho_peta_clip_b', 'ho_peta_bal05')}   # last = ConvNeXt
def sc(z):
    mm, _ = evaluate(1 / (1 + np.exp(-z[m])), Y[m], src[m]); return [round(mm[k], 4) for k in ('Challenge_Avg', 'mA', 'Instance_F1')]
for r, z in Z.items(): print(f'{r:16s}', sc(z))
for combo in (('ho_peta_clip_a', 'ho_peta_clip_b'), ('ho_peta_clip_b', 'ho_peta_bal05'), ('ho_peta_clip_a', 'ho_peta_clip_b', 'ho_peta_bal05')):
    print(' + '.join(combo), sc(np.mean([Z[c] for c in combo], axis=0)))

### Threshold tuning, ConvNeXt-Tiny

Per-attribute, per-group and one global threshold, fitted on one held-out source and scored on the other. Only a global shift helped, and only by about 0.003.

In [ ]:
import sys, json, numpy as np, pandas as pd
from pathlib import Path
sys.path.insert(0, '/content/drive/MyDrive/upar/code')
from upar_eval import _metrics
DRIVE = Path('/content/drive/MyDrive/upar'); RUNS = DRIVE / 'runs'
va = pd.read_csv('/content/data/annotations/task1/val/gt.csv')
NAMES = list(va.columns[1:]); Y = va[NAMES].to_numpy(np.uint8)
SRC = va.iloc[:, 0].str.split('/').str[0].to_numpy()

RECIPES = {   # members whose logits are averaged; the first scenario is PETA held out, the second Market1501
    'bal05': {'PETA': ['ho_peta_bal05'], 'Market1501': ['ho_market_bal05']},
    'ens3':  {'PETA': ['ho_peta_base8', 'ho_peta_bal05', 'ho_peta_bal10'],
              'Market1501': ['ho_market_base8', 'ho_market_bal05', 'ho_market_bal10']}}

def load_scenarios(recipe):
    out = []
    for held, runs in RECIPES[recipe].items():
        Z = np.mean([np.load(RUNS / r / 'val_logits_last.npy') for r in runs], axis=0)   # last epoch = the final models
        m = SRC == held
        out.append({'name': held, 'Z': Z[m].astype(np.float32), 'Y': Y[m], 'valid': Y[m].sum(0) > 0})
    return out

def score(sc, delta):                       # delta: logit-space threshold per attribute (0 = the usual 0.5 cut)
    pred = (sc['Z'] > delta).astype(np.float32)
    return _metrics(pred, sc['Y'], sc['valid'], 0.5)

def objective(scs, delta):
    return float(np.mean([score(s, delta)['Challenge_Avg'] for s in scs]))

for recipe in RECIPES:
    print(recipe, {s['name']: round(score(s, np.zeros(40))['Challenge_Avg'], 4) for s in load_scenarios(recipe)})

In [ ]:
GRID = np.arange(-3.0, 3.01, 0.25)
def idx(prefix): return [i for i, n in enumerate(NAMES) if n.startswith(prefix)]
GROUPS = [idx(p) for p in ('Age-', 'Gender-', 'Hair-Length-', 'UpperBody-Length-', 'UpperBody-Color-',
                           'LowerBody-Length-', 'LowerBody-Color-', 'LowerBody-Type-', 'Accessory-')]
assert sorted(sum(GROUPS, [])) == list(range(40))
VARIANTS = {'per-attribute': [[k] for k in range(40)], 'per-group': GROUPS, 'one global bias': [list(range(40))]}
LAMS = [0.0, 0.25, 0.5, 0.75, 1.0]

def coord_ascent(scs, groups, passes=3):
    d = np.zeros(40); best = objective(scs, d)
    for _ in range(passes):
        changed = False
        for g in groups:
            cur = best_v = d[g[0]]
            for v in GRID:
                if v == cur: continue
                d2 = d.copy(); d2[g] = v
                val = objective(scs, d2)
                if val > best + 1e-6: best, best_v = val, v
            if best_v != cur: d[g] = best_v; changed = True
        if not changed: break
    return d, best

def transfer_table(recipe):
    scs = load_scenarios(recipe); rows = []
    for vname, groups in VARIANTS.items():
        d_p, _ = coord_ascent([scs[0]], groups)       # tuned on PETA held out
        d_m, _ = coord_ascent([scs[1]], groups)       # tuned on Market held out
        rows.append({'variant': vname, **{f'lam={l}': (objective([scs[1]], l * d_p) + objective([scs[0]], l * d_m)) / 2 for l in LAMS}})
    return pd.DataFrame(rows).set_index('variant')

TT = {r: transfer_table(r) for r in RECIPES}
for r, t in TT.items():
    print(f'\n=== {r}: score on the OTHER held-out source after tuning on one (lam = shrinkage towards 0.5; lam=0 is uncalibrated) ===')
    display(t.round(4))

In [ ]:
MIN_GAIN = 0.002                  # if no setting beats the uncalibrated cut by this much on the other source, keep 0.5
def sigmoid(z): return 1 / (1 + np.exp(-z))

def fit_final(recipe):
    scs, tt = load_scenarios(recipe), TT[recipe]
    base = tt['lam=0.0'].iloc[0]
    v, l, s = max(((v, l, tt.loc[v, f'lam={l}']) for v in tt.index for l in LAMS), key=lambda x: x[2])
    if s - base < MIN_GAIN:
        print(f'{recipe}: best transfer gain {s - base:+.4f} < {MIN_GAIN}: keeping thresholds at 0.5'); d = np.zeros(40); v, l = 'none', 0.0
    else:
        d, _ = coord_ascent(scs, VARIANTS[v]); d = l * d
        print(f'{recipe}: variant "{v}", lam {l}, transfer gain {s - base:+.4f}')
    t = np.clip(sigmoid(d), 1e-6, 1 - 1e-6)
    path = DRIVE / f'thresholds_{recipe}.json'
    path.write_text(json.dumps({'attribute_names': NAMES, 'thresholds': t.tolist(), 'recipe': recipe, 'variant': v, 'lambda': l}))
    rep = [{'scenario': sc['name'], 'cut': name, **{k: round(score(sc, dd)[k], 4) for k in ('mA', 'Instance_F1', 'Challenge_Avg')}}
           for sc in scs for name, dd in (('0.5', np.zeros(40)), ('tuned (in-sample)', d))]
    display(pd.DataFrame(rep))
    tab = pd.DataFrame({'attribute': NAMES, 'delta (logit)': d, 'threshold t_k': t}).round(3)
    display(tab[tab['delta (logit)'] != 0])
    return t, path

THR = {r: fit_final(r) for r in RECIPES}

# unit test: p = sigmoid(z - logit(t_k)) clipped; p > 0.5  <=>  sigmoid(z) > t_k
rng = np.random.default_rng(0)
for r, (t, _) in THR.items():
    Zr = rng.normal(0, 4, (200000, 40)); d = np.log(t / (1 - t))
    p = np.clip(sigmoid(Zr - d), 1e-6, 1 - 1e-6)
    bad = ((p > 0.5) != (sigmoid(Zr) > t)) & (np.abs(Zr - d) > 1e-6)
    assert bad.sum() == 0 and p.min() >= 1e-6 and p.max() <= 1 - 1e-6, (r, int(bad.sum()))
print('\nunit test passed: p > 0.5 iff sigmoid(z) > t_k, and p stays in [1e-6, 1-1e-6]')

### Threshold tuning, CLIP ensemble

No shift gained 0.002 or more, so the final model keeps the plain 0.5 cut.

In [ ]:
import sys, json, numpy as np, pandas as pd
from pathlib import Path
sys.path.insert(0, '/content/drive/MyDrive/upar/code')
from upar_eval import _metrics
DRIVE = Path('/content/drive/MyDrive/upar'); RUNS = DRIVE / 'runs'
va = pd.read_csv('/content/data/annotations/task1/val/gt.csv'); NAMES = list(va.columns[1:])
Y = va[NAMES].to_numpy(np.uint8); src = va.iloc[:, 0].str.split('/').str[0].to_numpy(); msk = src == 'PETA'
z = np.mean([np.load(RUNS / r / 'val_logits_last.npy') for r in ('ho_peta_clip_a', 'ho_peta_clip_b')], axis=0)[msk]
y = Y[msk]; valid = y.sum(0) > 0
tab = []
for d in np.arange(-1.0, 1.001, 0.125):
    r = _metrics((z > d).astype(np.float32), y, valid, 0.5)
    tab.append({'delta': round(float(d), 3), 'threshold': round(float(1 / (1 + np.exp(-d))), 3),
                **{k: round(float(r[k]), 4) for k in ('Challenge_Avg', 'mA', 'Instance_F1', 'Instance_Prec', 'Instance_Rec')}})
tab = pd.DataFrame(tab); display(tab)
base = tab.loc[np.isclose(tab.delta, 0), 'Challenge_Avg'].item(); best = tab.loc[tab.Challenge_Avg.idxmax()]
LAM, MIN_GAIN = 0.5, 0.002
d_final = LAM * best.delta if best.Challenge_Avg - base >= MIN_GAIN else 0.0
TH = DRIVE / 'thresholds_clip.json'
if d_final != 0:
    TH.write_text(json.dumps({'attribute_names': NAMES, 'thresholds': [float(1 / (1 + np.exp(-d_final)))] * 40}))
    print(f'global shift {d_final:+.3f} (threshold {1 / (1 + np.exp(-d_final)):.3f}) saved; held-out gain at full shift {best.Challenge_Avg - base:+.4f}')
else:
    TH.unlink(missing_ok=True); print(f'no shift: best gain {best.Challenge_Avg - base:+.4f} < {MIN_GAIN}; thresholds stay 0.5')

Flip test-time augmentation was also tested on the full validation set with a ConvNeXt-Tiny model: Challenge_Avg 0.8529 without it and 0.8539 with it. The final `run.py` uses it behind a time guard.

## Part 6. Codabench submission

Package the models as the platform expects (`run.py` and `metadata.yaml` at the root of the zip) and test the package the way the platform runs it.

### Submission 1: the sample baseline

An unchanged copy of the organisers' example, to check the upload and scoring path. It scored 0.4955.

In [ ]:
import shutil, zipfile, hashlib
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/upar')

src = next(Path('/content/repo_src').glob('**/examples/task1/sample_code_submission'), None)
if src is None:                                   # fresh session: re-extract the repo zip from Drive
    repo_zip = next(p for p in DRIVE.glob('*.zip') if 'upar' in p.name.lower())
    with zipfile.ZipFile(repo_zip) as zf:
        zf.extractall('/content/repo_src')
    src = next(Path('/content/repo_src').glob('**/examples/task1/sample_code_submission'))

SUB = Path('/content/submission_baseline')
shutil.rmtree(SUB, ignore_errors=True)
shutil.copytree(src, SUB, ignore=shutil.ignore_patterns('__pycache__', '.DS_Store', '.git'))

EXPECTED = {   # sha256 of the original files in the repo I read
    'run.py': '6f1a2e1d709a0a379c854371c8903201551a32a86ca02f2a35881ffbdec82298',
    'metadata.yaml': 'b5b7f62f63049241c2e4e7ea4cf7d350cb7c5d427335cb1bbedcda9a1b8b397a',
    'assets/attribute_prior.json': '191213e67537db28b00d5aaecb6bb0f04e16cd0623ee51ab706aff175b191c59'}
for rel, want in EXPECTED.items():
    got = hashlib.sha256((SUB / rel).read_bytes()).hexdigest()
    print(f'{rel:30s}', 'unmodified' if got == want else 'DIFFERS from the original (edited, or line endings changed?)')

In [ ]:
%%bash
set -e
which zip >/dev/null || apt-get -qq install -y zip
cd /content/submission_baseline
rm -f /content/submission_baseline.zip
zip -r /content/submission_baseline.zip . -x '*/__pycache__/*' '*.DS_Store' '.git/*'

In [ ]:
ZIP = Path('/content/submission_baseline.zip')
!unzip -l /content/submission_baseline.zip
with zipfile.ZipFile(ZIP) as zf:
    names = zf.namelist()
    assert zf.testzip() is None, 'corrupt zip'
    assert 'run.py' in names and 'metadata.yaml' in names, 'run.py / metadata.yaml are NOT at the archive root'
    assert not any(n.startswith(('sample_code_submission/', './')) for n in names), 'extra top-level folder inside the zip'
    assert not any('__pycache__' in n or n.endswith('.DS_Store') or n.startswith('.git/') for n in names), 'junk files in the zip'
    meta_same = zf.read('metadata.yaml') == (SUB / 'metadata.yaml').read_bytes()
print(f'\nOK: run.py and metadata.yaml at the root | metadata.yaml identical to source: {meta_same} | size {ZIP.stat().st_size / 1e3:.1f} kB')

In [ ]:
import csv, importlib.util, os, sys, time, json
import numpy as np

def to_vector(out, names):
    if isinstance(out, dict):
        missing = [n for n in names if n not in out]
        assert not missing, f'dict output misses {missing[:3]}'
        out = [out[n] for n in names]
    if hasattr(out, 'detach'):
        out = out.detach().cpu().numpy()
    v = np.asarray(out, dtype=np.float64)
    assert v.shape == (len(names),), f'wrong length: {v.shape}, expected ({len(names)},)'
    assert np.isfinite(v).all(), 'NaN or inf in output'
    assert v.min() >= 0.0 and v.max() <= 1.0, f'values outside [0,1]: {v.min()}..{v.max()}'
    return v

def simulate_ingestion(folder, names, n=100):
    folder = Path(folder).resolve()
    old_cwd, old_flag = os.getcwd(), sys.dont_write_bytecode
    os.chdir(folder); sys.path.insert(0, str(folder)); sys.dont_write_bytecode = True
    try:
        spec = importlib.util.spec_from_file_location('submission_run', folder / 'run.py')
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        t = time.time()
        if hasattr(mod, 'load_model'):
            mod.load_model()
        print(f'load_model: {time.time() - t:.2f} s')
        samples = [{'index': i, 'image': f'fake_dataset/images/{i}.jpg',
                    'image_path': f'/app/input_data/images/fake_dataset/images/{i}.jpg',
                    'attribute_names': list(names)} for i in range(n)]
        t = time.time(); outs = []
        if hasattr(mod, 'predict_batch'):
            bs = getattr(mod, 'BATCH_SIZE', 32)
            print(f'using predict_batch, BATCH_SIZE={bs}')
            for i in range(0, n, bs):
                chunk = samples[i:i + bs]
                res = list(mod.predict_batch(chunk))
                assert len(res) == len(chunk), f'predict_batch returned {len(res)} results for {len(chunk)} samples'
                outs += res
        else:
            print('using predict_image')
            outs = [mod.predict_image(s) for s in samples]
        dt = time.time() - t
        P = np.stack([to_vector(o, names) for o in outs])
        print(f'{n} samples OK in {dt:.3f} s ({1000 * dt / n:.2f} ms/image); output {P.shape}, min {P.min():.3f}, max {P.max():.3f}')
        print('attributes predicted present (>0.5):', [nm for nm, p in zip(names, P[0]) if p > 0.5])
        return P
    finally:
        os.chdir(old_cwd); sys.path.remove(str(folder)); sys.dont_write_bytecode = old_flag
        sys.modules.pop('submission_run', None)

# attribute order exactly as in gt.csv (the platform sends the same 40 names)
with open(src.parents[2] / 'data/annotations/task1/val/gt.csv', newline='') as f:
    NAMES = next(csv.reader(f))[1:]
assert len(NAMES) == 40

TEST = Path('/content/ingest_test'); shutil.rmtree(TEST, ignore_errors=True)
with zipfile.ZipFile(ZIP) as zf:
    zf.extractall(TEST)
prior_names = json.loads((TEST / 'assets/attribute_prior.json').read_text())['attribute_names']
print('prior names identical to gt.csv order:', prior_names == NAMES)
P = simulate_ingestion(TEST, NAMES)
print('\nINGESTION SIMULATION PASSED')

In [ ]:
from google.colab import files
(DRIVE / 'submissions').mkdir(exist_ok=True)
shutil.copyfile(ZIP, DRIVE / 'submissions' / '01_baseline_prior.zip')   # small; keeps a copy of everything you submit
print('backup:', DRIVE / 'submissions' / '01_baseline_prior.zip')
files.download(str(ZIP))

### `run.py` for the final submission

Averages the logits of one or more models (CLIP B/16 or L/14, fp16 or int8 weights), cuts at 0.5, never crashes on a bad image, and drops flip TTA if the run would get too slow for the 5-minute limit. This version also runs the earlier B/16 package.

In [ ]:
import os
os.makedirs('/content/run_src', exist_ok=True)   # %%writefile cannot create the folder in a fresh session

In [ ]:
%%writefile /content/run_src/run.py
"""UPAR 2027 Track 1 submission: weighted ensemble of one or more models (CLIP ViT via the shipped upar_clipvit.py,
or torchvision), logits averaged, optional horizontal-flip TTA with a time guard.

Only torch, torchvision, numpy and Pillow are used. Nothing here looks at sample['image'] or at the directory of
sample['image_path']: the same models and thresholds are used for every image.
"""
import atexit
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
import torch.nn as nn
import torchvision
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

HERE = os.path.dirname(os.path.abspath(__file__))
ASSETS = os.path.join(HERE, "assets")
BATCH_SIZE = 64

USE_TTA = True             # horizontal-flip TTA; switched off automatically when throughput is too low
N_ASSUMED = 40000          # test-set size assumed by the time guard (the platform log showed 29,248 images)
TIME_BUDGET_S = 220.0      # of the 300 s limit; the rest is margin
N_THREADS = 12

PAD_RGB = (124, 116, 104)  # must equal upar_preproc.PAD_RGB used to build the training cache
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
_S = {}


def _log(msg):
    print(f"[run.py] {msg}", flush=True)


def letterbox(img, H, W, mode="pad"):
    """Identical to upar_preproc.letterbox (training cache): keep aspect ratio, centre on a mean-coloured canvas."""
    img = img.convert("RGB")
    w, h = img.size
    if mode == "stretch":
        return np.asarray(img.resize((W, H), Image.BICUBIC), dtype=np.uint8)
    s = min(W / w, H / h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    img = img.resize((nw, nh), Image.BICUBIC)
    if (nw, nh) == (W, H):
        return np.asarray(img, dtype=np.uint8)
    canvas = Image.new("RGB", (W, H), PAD_RGB)
    canvas.paste(img, ((W - nw) // 2, (H - nh) // 2))
    return np.asarray(canvas, dtype=np.uint8)


def _build(mcfg, n):
    arch = mcfg["arch"]
    if arch == "convnext_tiny":
        m = torchvision.models.convnext_tiny(weights=None)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, n)
        return m
    if arch == "resnet50":
        m = torchvision.models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, n)
        return m
    if arch.startswith("clip"):
        if HERE not in sys.path:
            sys.path.insert(0, HERE)
        from upar_clipvit import ClipPAR            # shipped next to run.py; plain torch, no open_clip
        dims = {k: mcfg[k] for k in ("patch", "width", "layers", "heads", "out_dim") if k in mcfg}
        return ClipPAR(tuple(mcfg["in_size"]), mcfg["act"], n, **dims)
    raise ValueError(arch)


def _dequant(sd):
    """int8 weights are stored as <name>::q (int8) plus <name>::s (one fp16 scale per row)."""
    out = {}
    for k, v in sd.items():
        if k.endswith("::s"):
            continue
        if k.endswith("::q"):
            base = k[:-3]
            s = sd[base + "::s"].float()
            out[base] = (v.float().reshape(v.shape[0], -1) * s.reshape(-1, 1)).reshape(v.shape)
        else:
            out[k] = v
    return out


def load_model():
    t0 = time.time()
    with open(os.path.join(ASSETS, "config.json")) as f:
        cfg = json.load(f)
    names = list(cfg["attribute_names"])
    n = len(names)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    members = cfg.get("members") or [dict(cfg, file="model.pth")]
    models = []
    for mcfg in members:
        model = _build(mcfg, n)
        sd = torch.load(os.path.join(ASSETS, mcfg["file"]), map_location="cpu", weights_only=True)
        model.load_state_dict(_dequant(sd))
        del sd
        model = model.eval().to(dev)
        if dev == "cuda":
            model = model.to(memory_format=torch.channels_last)
        mean = torch.tensor(mcfg.get("mean", MEAN), device=dev).view(1, 3, 1, 1)
        std = torch.tensor(mcfg.get("std", STD), device=dev).view(1, 3, 1, 1)
        models.append((model, mean, std, float(mcfg.get("weight", 1.0))))
    if dev == "cuda":
        torch.backends.cudnn.benchmark = True

    thr = np.full(n, 0.5)
    tpath = os.path.join(ASSETS, "thresholds.json")
    if os.path.exists(tpath):
        with open(tpath) as f:
            td = json.load(f)
        m = dict(zip(td["attribute_names"], td["thresholds"]))
        thr = np.array([m.get(nm, 0.5) for nm in names], dtype=np.float64)
    thr = np.clip(thr, 1e-6, 1 - 1e-6)
    with open(os.path.join(ASSETS, "prior.json")) as f:
        pr = json.load(f)
    pm = dict(zip(pr["attribute_names"], pr["prior"]))
    _S.update(
        models=models, dev=dev, names=names, pos={nm: i for i, nm in enumerate(names)}, cols={},
        H=int(cfg["H"]), W=int(cfg["W"]), mode=cfg.get("mode", "pad"),
        bias=torch.tensor(np.log(thr / (1 - thr)), dtype=torch.float32, device=dev),
        prior=np.array([pm.get(nm, 0.0) for nm in names], dtype=np.float64),
        pool=ThreadPoolExecutor(N_THREADS), tta=bool(USE_TTA), t_boot=t0, t_first=None, n_done=0, n_batches=0,
        n_fail=0, has_thr=os.path.exists(tpath))
    _log(f"{len(models)} model(s) {[mc['arch'] for mc in members]} on {dev}; thresholds file: {_S['has_thr']}; "
         f"TTA: {_S['tta']}; load {time.time() - t0:.1f}s")
    atexit.register(_summary)


def _summary():
    if _S.get("t_first") is None:
        return
    dt = time.time() - _S["t_first"]
    _log(f"DONE {_S['n_done']} images in {dt:.1f}s ({_S['n_done'] / max(dt, 1e-9):.0f} img/s), "
         f"unreadable: {_S['n_fail']}, TTA at end: {_S['tta']}")


def _load(path):
    try:
        with Image.open(path) as im:
            return letterbox(im, _S["H"], _S["W"], _S["mode"])
    except Exception as e:
        return e


@torch.no_grad()
def _forward(x_uint8):
    dev = _S["dev"]
    x0 = x_uint8.to(dev).permute(0, 3, 1, 2).float().div_(255)
    z, wsum = 0.0, 0.0
    with torch.autocast(dev, dtype=torch.float16, enabled=(dev == "cuda")):
        for model, mean, std, w in _S["models"]:
            x = (x0 - mean) / std
            if dev == "cuda":
                x = x.contiguous(memory_format=torch.channels_last)
            zm = model(x).float()
            if _S["tta"]:
                zm = (zm + model(x.flip(3)).float()) / 2
            z = z + w * zm
            wsum += w
    z = z / wsum
    p = torch.sigmoid(z - _S["bias"]).clamp_(1e-6, 1 - 1e-6)       # p > 0.5  <=>  sigmoid(z) > t_k
    return p.cpu().numpy().astype(np.float64)


def _columns(names):
    key = tuple(names)
    if key not in _S["cols"]:
        _S["cols"][key] = [_S["pos"].get(n, -1) for n in names]
        if -1 in _S["cols"][key]:
            _log(f"unknown attribute names (answered with 0.0): {[n for n in names if n not in _S['pos']]}")
    return _S["cols"][key]


def _time_guard():
    now = time.time()
    if _S["n_batches"] % 10 or _S["n_done"] < 640:
        return
    rate = _S["n_done"] / max(now - _S["t_first"], 1e-9)
    projected = (now - _S["t_boot"]) + (N_ASSUMED - _S["n_done"]) / rate
    if _S["tta"] and projected > TIME_BUDGET_S:
        _S["tta"] = False
        _log(f"TTA disabled: {rate:.0f} img/s projects {projected:.0f}s for {N_ASSUMED} images (budget {TIME_BUDGET_S:.0f}s)")
    if _S["n_batches"] % 50 == 0:
        _log(f"{_S['n_done']} images, {rate:.0f} img/s, projected total {projected:.0f}s, TTA {_S['tta']}")


def predict_batch(samples):
    if "models" not in _S:
        load_model()
    if _S["t_first"] is None:
        _S["t_first"] = time.time()
    names = list(samples[0]["attribute_names"]) if samples else list(_S["names"])
    cols = _columns(names)
    paths = [s["image_path"] for s in samples]
    arrs = list(_S["pool"].map(_load, paths))

    P = np.tile(_S["prior"], (len(samples), 1))                    # fallback for unreadable images
    ok = [i for i, a in enumerate(arrs) if isinstance(a, np.ndarray)]
    for i, a in enumerate(arrs):
        if not isinstance(a, np.ndarray):
            _S["n_fail"] += 1
            if _S["n_fail"] <= 20:
                _log(f"unreadable image, using the training prior: {paths[i]} ({a!r})")
    if ok:
        try:
            P[ok] = _forward(torch.from_numpy(np.stack([arrs[i] for i in ok])))
        except Exception as e:                                      # never crash the whole run
            _log(f"inference failed for a batch, using the training prior: {e!r}")
    bad = ~np.isfinite(P).all(axis=1)
    if bad.any():
        _log(f"{int(bad.sum())} non-finite rows replaced by the training prior")
        P[bad] = _S["prior"]
    P = np.clip(np.nan_to_num(P, nan=0.5), 0.0, 1.0)

    out = np.zeros((len(samples), len(names)), dtype=np.float64)    # map by NAME to the requested order
    for j, c in enumerate(cols):
        if c >= 0:
            out[:, j] = P[:, c]
    _S["n_done"] += len(samples)
    _S["n_batches"] += 1
    _time_guard()
    return out.tolist()

### Package and check the three-model CLIP B/16 ensemble (submission 4, 0.7694)

Builds the zip from the three B/16 models (fp16), runs `run.py` on 500 validation images without `open_clip` or `timm`, and compares its output with the training pipeline.

In [ ]:
import sys, os, json, shutil, zipfile, subprocess, time, importlib.util
import numpy as np, pandas as pd, torch
from pathlib import Path
DRIVE, DATA = Path('/content/drive/MyDrive/upar'), Path('/content/data'); RUNS = DRIVE / 'runs'
MEMBERS = ['final_clip_b', 'final_clip_a', 'final_clip_b_s1']     # drop the last one if the zip is too big
sys.path.insert(0, str(DRIVE / 'code'))
from upar_clipvit import ClipPAR
from upar_data import GPUAugment

tr = pd.read_csv(DATA / 'annotations/task1/train/gt.csv'); va = pd.read_csv(DATA / 'annotations/task1/val/gt.csv')
NAMES = list(tr.columns[1:])
src_sample = next(Path('/content/repo_src').glob('**/examples/task1/sample_code_submission'))
pkg = Path('/content/pkg_ens'); shutil.rmtree(pkg, ignore_errors=True); (pkg / 'assets').mkdir(parents=True)
members, sds = [], []
for i, r in enumerate(MEMBERS):
    meta = json.loads((RUNS / r / 'meta.json').read_text()); mc = meta['model_cfg']
    assert meta['cfg']['arch'] == 'clip_b16' and meta['attribute_names'] == NAMES, r
    sd = torch.load(RUNS / r / 'final_state_dict.pt', map_location='cpu', weights_only=True)
    sd16 = {k: (v.half() if v.is_floating_point() else v) for k, v in sd.items()}
    torch.save(sd16, pkg / 'assets' / f'model_{i}.pth'); sds.append(sd16)
    members.append({'arch': 'clip_b16', 'file': f'model_{i}.pth', 'in_size': mc['in_size'], 'act': mc['act'], 'mean': mc['mean'], 'std': mc['std']})
    print('member', i, r)
(pkg / 'assets' / 'config.json').write_text(json.dumps({'H': 192, 'W': 96, 'mode': 'pad', 'attribute_names': NAMES, 'members': members}))
(pkg / 'assets' / 'prior.json').write_text(json.dumps({'attribute_names': NAMES, 'prior': tr[NAMES].mean().tolist()}))
TH = DRIVE / 'thresholds_clip.json'
if TH.exists(): shutil.copy(TH, pkg / 'assets' / 'thresholds.json')
shutil.copy(src_sample / 'metadata.yaml', pkg / 'metadata.yaml')
shutil.copy('/content/run_src/run.py', pkg / 'run.py')
shutil.copy(DRIVE / 'code' / 'upar_clipvit.py', pkg / 'upar_clipvit.py')
assert "members" in (pkg / 'run.py').read_text(), 'run cell 2 (run.py v3) first'
ZIP = Path('/content/submission_clip_ens.zip'); ZIP.unlink(missing_ok=True)
subprocess.run('which zip >/dev/null || apt-get -qq install -y zip', shell=True)
subprocess.run(['zip', '-r', str(ZIP), '.', '-x', '*/__pycache__/*', '*.DS_Store', '.git/*'], cwd=pkg, check=True, capture_output=True)
with zipfile.ZipFile(ZIP) as zf: names = zf.namelist()
assert {'run.py', 'metadata.yaml', 'upar_clipvit.py', 'assets/config.json', 'assets/prior.json'} <= set(names), names
print(f'{ZIP.name}: {ZIP.stat().st_size / 1e6:.0f} MB, thresholds file: {TH.exists()}')

def fake_ingestion(zip_path, rows, use_tta, workdir='/content/ingest_ens'):
    shutil.rmtree(workdir, ignore_errors=True)
    with zipfile.ZipFile(zip_path) as zf: zf.extractall(workdir)
    folder, old_cwd, code = Path(workdir), os.getcwd(), str(DRIVE / 'code')
    saved = {m: sys.modules.pop(m, None) for m in ('open_clip', 'timm', 'upar_clipvit')}
    sys.modules['open_clip'] = None; sys.modules['timm'] = None
    in_path = code in sys.path
    if in_path: sys.path.remove(code)
    os.chdir(folder); sys.dont_write_bytecode = True
    try:
        spec = importlib.util.spec_from_file_location('submission_run', folder / 'run.py')
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        mod.USE_TTA = use_tta
        t = time.time(); mod.load_model(); t_load = time.time() - t
        assert sys.modules['upar_clipvit'].__file__.startswith(str(folder)), 'run.py did not use the shipped module'
        samples = [{'index': int(i), 'image': va.iloc[int(i), 0], 'image_path': str(DATA / va.iloc[int(i), 0]),
                    'attribute_names': list(NAMES)} for i in rows]
        out, t = [], time.time()
        for b in range(0, len(samples), mod.BATCH_SIZE):
            chunk = samples[b:b + mod.BATCH_SIZE]; res = mod.predict_batch(chunk); assert len(res) == len(chunk)
            for r in res:
                v = np.asarray(r, dtype=np.float64)
                assert v.shape == (40,) and np.isfinite(v).all() and v.min() >= 0 and v.max() <= 1, 'bad output'
                out.append(v)
        return np.stack(out), time.time() - t, t_load
    finally:
        os.chdir(old_cwd)
        for m in ('open_clip', 'timm', 'upar_clipvit'): sys.modules.pop(m, None)
        for m, v in saved.items():
            if v is not None: sys.modules[m] = v
        if in_path: sys.path.insert(0, code)
        sys.modules.pop('submission_run', None)

rows = np.sort(np.random.default_rng(0).choice(len(va), 500, replace=False))
P, dt, t_load = fake_ingestion(ZIP, rows, use_tta=False)
print(f'no TTA: load {t_load:.1f}s, {500 / dt:.0f} img/s here')

# reference: TRAINING data path, mean of the members' logits, same fp16 weights, same threshold shift
X = np.load('/content/cache/h192w96_pad/val_images.npy', mmap_mode='r')
aug = GPUAugment(device='cuda')
aug.mean = torch.tensor(members[0]['mean'], device='cuda').view(1, 3, 1, 1); aug.std = torch.tensor(members[0]['std'], device='cuda').view(1, 3, 1, 1)
zs = []
for mcfg, sd16 in zip(members, sds):
    mdl = ClipPAR(tuple(mcfg['in_size']), mcfg['act'], 40)
    mdl.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in sd16.items()}); mdl = mdl.cuda().eval()
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        zs.append(torch.cat([mdl(aug.eval(torch.from_numpy(np.ascontiguousarray(X[rows[i:i + 100]])).cuda())).float().cpu()
                             for i in range(0, len(rows), 100)]).numpy())
    del mdl; torch.cuda.empty_cache()
z = np.mean(zs, axis=0)
t = np.array(json.loads(TH.read_text())['thresholds']) if TH.exists() else np.full(40, 0.5)
ref = 1 / (1 + np.exp(-(z - np.log(t / (1 - t)))))
d = np.abs(P - ref); agree = ((P > .5) == (ref > .5)).mean()
print(f'vs training path: max |dp| {d.max():.4f}, mean {d.mean():.5f}, same decision {agree * 100:.2f}%')
assert d.max() < 0.05 and agree > 0.995, 'MISMATCH: do not submit, send me this output'

P2, dt2, _ = fake_ingestion(ZIP, rows, use_tta=True)
print(f'with TTA: {500 / dt2:.0f} img/s here -> 29,248 images ~{29248 / (500 / dt2) / 60:.1f} min on this T4 (grader A10 is faster; the time guard drops TTA if needed)')
shutil.copyfile(ZIP, DRIVE / 'submissions' / '05_clip_ens.zip')
from google.colab import files; files.download(str(ZIP))
print('READY TO SUBMIT')

## Part 7. CLIP ViT-L/14 submissions

The same packaging code, now for L/14 models (`MEMBERS` lists run folders and weights). Two zips were built from the two final runs:

| Submission | Zip | What | Score |
|---|---|---|---|
| 5 | `submission_l14.zip` | `final_l14_a` alone, fp16 weights | 0.7788 (mA 0.7802, Instance_F1 0.7774), grader time 141 s |
| 6 | `06_submission_l14_ensemble.zip` | `final_l14_a` + `final_l14_lpft`, int8 weights | 0.7773 (mA 0.7754, Instance_F1 0.7792) |

Submission 6 was the last one allowed, so 0.7773 is the final score. The cell below is the one for the ensemble; for submission 5 set `MEMBERS = [('final_l14_a', 1.0)]`, `INT8 = False` and `ZIP = Path('/content/submission_l14.zip')`. The `run.py` cell above must have been run first.

In [ ]:
import sys, os, json, shutil, zipfile, subprocess, time, importlib, importlib.util
import numpy as np, pandas as pd, torch
from pathlib import Path
DRIVE, DATA = Path('/content/drive/MyDrive/upar'), Path('/content/data'); RUNS = DRIVE / 'runs'
# MEMBERS = [('final_l14_a', 1.0)]          # (run folder in upar/runs, weight)
MEMBERS = [('final_l14_a', 1.0), ('final_l14_lpft', 1.0)]
# INT8 = False                              # True = int8 weights, half the size
INT8 = True
# ZIP = Path('/content/submission_l14.zip')
ZIP = Path('/content/submission_l14_ensemble.zip')
DIMS = ('patch', 'width', 'layers', 'heads', 'out_dim')
sys.path.insert(0, str(DRIVE / 'code'))
import upar_clipvit; importlib.reload(upar_clipvit)
from upar_clipvit import ClipPAR
from upar_data import GPUAugment

tr = pd.read_csv(DATA / 'annotations/task1/train/gt.csv'); va = pd.read_csv(DATA / 'annotations/task1/val/gt.csv')
NAMES = list(tr.columns[1:])
src_sample = next(Path('/content/repo_src').glob('**/examples/task1/sample_code_submission'))

def load_weights(run):
    for f in ('final_fp16.pt', 'final_state_dict.pt'):
        if (RUNS / run / f).exists():
            return torch.load(RUNS / run / f, map_location='cpu', weights_only=True)
    raise FileNotFoundError(f'no weights in {RUNS / run}')

def quant_int8(sd):
    out = {}
    for k, v in sd.items():
        if v.is_floating_point() and v.ndim >= 2 and v.numel() >= 65536:
            w = v.float().reshape(v.shape[0], -1)
            s = (w.abs().amax(1) / 127.0).clamp(min=1e-8)
            out[k + '::q'] = torch.round(w / s[:, None]).clamp(-127, 127).to(torch.int8).reshape(v.shape)
            out[k + '::s'] = s.half()
        else:
            out[k] = v.half() if v.is_floating_point() else v
    return out

def dequant(sd):                          # same as run.py
    out = {}
    for k, v in sd.items():
        if k.endswith('::s'):
            continue
        if k.endswith('::q'):
            base = k[:-3]; s = sd[base + '::s'].float()
            out[base] = (v.float().reshape(v.shape[0], -1) * s.reshape(-1, 1)).reshape(v.shape)
        else:
            out[k] = v
    return out

pkg = Path('/content/pkg_new'); shutil.rmtree(pkg, ignore_errors=True); (pkg / 'assets').mkdir(parents=True)
members, ref_sds, fp_sds = [], [], []
for i, (run, wgt) in enumerate(MEMBERS):
    meta = json.loads((RUNS / run / 'meta.json').read_text()); mc = meta['model_cfg']
    assert meta['cfg']['arch'].startswith('clip') and meta['attribute_names'] == NAMES, run
    sd16 = {k: (v.half() if v.is_floating_point() else v) for k, v in load_weights(run).items()}
    stored = quant_int8(sd16) if INT8 else sd16
    torch.save(stored, pkg / 'assets' / f'model_{i}.pth')
    ref_sds.append(dequant(stored)); fp_sds.append(sd16)
    members.append({'arch': mc['arch'], 'file': f'model_{i}.pth', 'weight': float(wgt), 'in_size': mc['in_size'],
                    'act': mc['act'], 'mean': mc['mean'], 'std': mc['std'], **{k: mc[k] for k in DIMS if k in mc}})
    print(f'member {i}: {run} ({mc["arch"]}, weight {wgt}), {(pkg / "assets" / f"model_{i}.pth").stat().st_size / 1e6:.0f} MB')
(pkg / 'assets' / 'config.json').write_text(json.dumps({'H': 192, 'W': 96, 'mode': 'pad', 'attribute_names': NAMES, 'members': members}))
(pkg / 'assets' / 'prior.json').write_text(json.dumps({'attribute_names': NAMES, 'prior': tr[NAMES].mean().tolist()}))
shutil.copy(src_sample / 'metadata.yaml', pkg / 'metadata.yaml')
shutil.copy('/content/run_src/run.py', pkg / 'run.py')
shutil.copy(DRIVE / 'code' / 'upar_clipvit.py', pkg / 'upar_clipvit.py')
assert '_dequant' in (pkg / 'run.py').read_text(), 'run cell L6 (run.py v4) first'
ZIP.unlink(missing_ok=True)
subprocess.run('which zip >/dev/null || apt-get -qq install -y zip', shell=True)
subprocess.run(['zip', '-r', str(ZIP), '.', '-x', '*/__pycache__/*', '*.DS_Store', '.git/*'], cwd=pkg, check=True, capture_output=True)
with zipfile.ZipFile(ZIP) as zf: names = zf.namelist()
assert {'run.py', 'metadata.yaml', 'upar_clipvit.py', 'assets/config.json', 'assets/prior.json'} <= set(names), names
print(f'{ZIP.name}: {ZIP.stat().st_size / 1e6:.0f} MB')

def fake_ingestion(zip_path, rows, use_tta, workdir='/content/ingest_new'):
    shutil.rmtree(workdir, ignore_errors=True)
    with zipfile.ZipFile(zip_path) as zf: zf.extractall(workdir)
    folder, old_cwd, code = Path(workdir), os.getcwd(), str(DRIVE / 'code')
    saved = {m: sys.modules.pop(m, None) for m in ('open_clip', 'timm', 'upar_clipvit')}
    sys.modules['open_clip'] = None; sys.modules['timm'] = None
    in_path = code in sys.path
    if in_path: sys.path.remove(code)
    os.chdir(folder); sys.dont_write_bytecode = True
    try:
        spec = importlib.util.spec_from_file_location('submission_run', folder / 'run.py')
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        mod.USE_TTA = use_tta
        t = time.time(); mod.load_model(); t_load = time.time() - t
        assert sys.modules['upar_clipvit'].__file__.startswith(str(folder)), 'run.py did not use the shipped module'
        samples = [{'index': int(i), 'image': va.iloc[int(i), 0], 'image_path': str(DATA / va.iloc[int(i), 0]),
                    'attribute_names': list(NAMES)} for i in rows]
        out, t = [], time.time()
        for b in range(0, len(samples), mod.BATCH_SIZE):
            chunk = samples[b:b + mod.BATCH_SIZE]; res = mod.predict_batch(chunk); assert len(res) == len(chunk)
            for r in res:
                v = np.asarray(r, dtype=np.float64)
                assert v.shape == (40,) and np.isfinite(v).all() and v.min() >= 0 and v.max() <= 1, 'bad output'
                out.append(v)
        return np.stack(out), time.time() - t, t_load
    finally:
        os.chdir(old_cwd)
        for m in ('open_clip', 'timm', 'upar_clipvit'): sys.modules.pop(m, None)
        for m, v in saved.items():
            if v is not None: sys.modules[m] = v
        if in_path: sys.path.insert(0, code)
        sys.modules.pop('submission_run', None)
        torch.cuda.empty_cache()

rows = np.sort(np.random.default_rng(0).choice(len(va), 500, replace=False))
P, dt, t_load = fake_ingestion(ZIP, rows, use_tta=False)
print(f'no TTA: load {t_load:.1f}s, {500 / dt:.0f} img/s here (T4)')

X = np.load('/content/cache/h192w96_pad/val_images.npy', mmap_mode='r')
aug = GPUAugment(device='cuda')
def member_logits(m, sd):
    aug.mean = torch.tensor(m['mean'], device='cuda').view(1, 3, 1, 1); aug.std = torch.tensor(m['std'], device='cuda').view(1, 3, 1, 1)
    mdl = ClipPAR(tuple(m['in_size']), m['act'], 40, **{k: m[k] for k in DIMS if k in m})
    mdl.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in sd.items()}); mdl = mdl.cuda().eval()
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        z = torch.cat([mdl(aug.eval(torch.from_numpy(np.ascontiguousarray(X[rows[i:i + 50]])).cuda())).float().cpu()
                       for i in range(0, len(rows), 50)]).numpy()
    del mdl; torch.cuda.empty_cache()
    return z
wsum = sum(m['weight'] for m in members)
z_ref = sum(m['weight'] * member_logits(m, sd) for m, sd in zip(members, ref_sds)) / wsum
ref = 1 / (1 + np.exp(-z_ref)); d = np.abs(P - ref); agree = ((P > .5) == (ref > .5)).mean()
print(f'vs training path: max |dp| {d.max():.4f}, mean {d.mean():.5f}, same decision {agree * 100:.2f}%')
assert d.max() < 0.05 and agree > 0.995, 'MISMATCH: do not submit, send me this output'
if INT8:
    z_fp = sum(m['weight'] * member_logits(m, sd) for m, sd in zip(members, fp_sds)) / wsum
    agree_q = ((z_ref > 0) == (z_fp > 0)).mean()
    print(f'int8 vs fp16 weights: same decision {agree_q * 100:.2f}%'); assert agree_q > 0.99, 'int8 changed too much: use INT8 = False'

P2, dt2, _ = fake_ingestion(ZIP, rows, use_tta=True)
print(f'with TTA: {500 / dt2:.0f} img/s here (the grader A10 is faster; the time guard drops TTA if needed)')
shutil.copyfile(ZIP, DRIVE / 'submissions' / ('06_' + ZIP.name))
from google.colab import files; files.download(str(ZIP))
print('READY TO SUBMIT')

member 0: final_l14_a (clip_l14, weight 1.0), 305 MB
member 1: final_l14_lpft (clip_l14, weight 1.0), 305 MB
submission_l14_ensemble.zip: 536 MB
[run.py] 2 model(s) ['clip_l14', 'clip_l14'] on cuda; thresholds file: False; TTA: False; load 8.2s
no TTA: load 8.2s, 71 img/s here (T4)
vs training path: max |dp| 0.0000, mean 0.00000, same decision 100.00%
int8 vs fp16 weights: same decision 99.94%
[run.py] 2 model(s) ['clip_l14', 'clip_l14'] on cuda; thresholds file: False; TTA: True; load 9.8s
with TTA: 54 img/s here (the grader A10 is faster; the time guard drops TTA if needed)


READY TO SUBMIT
